## Multi-Variable Leiden Clustering with Clustree

- Main ipynb using clustree visualizations as the primary way of dealing with leiden clusterings
- In practice, a much better representation of the data than UMAP, though not great for main figures
- Currently implemented visualizations:
    - Silhouette scores
    - Recall Rates (targetID, Gene)
    - Frequency in cluster (targetID, Gene)
    - % of target in cluster (targetID, Gene)
    - Observables (Z-score, Real values)
- Also have a tool at the end of the ipynb to extract a cosensus timeseries from any cluster
- Notes and a ZIP folder export function at the bottom of the ipynb
- **High Priority**: Use as the main tool for data exploration on any clustering
- Still need to go through this with the long and short datasets to make sure nothing major is missed

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr
import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances

from sklearn.metrics import silhouette_score
import scipy.stats
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering

import scanpy as sc
import anndata
import scipy as sp
import scipy.sparse
import dask.array as da
from tslearn.neighbors import KNeighborsTimeSeries
from tslearn.metrics import cdist_soft_dtw_normalized,cdist_soft_dtw
import networkx as nx
import igraph as ig
import leidenalg
import umap
from scanpy.plotting.palettes import default_20,vega_20_scanpy
from matplotlib import pyplot as plt
from matplotlib import cm
import ast
from tslearn.barycenters import softdtw_barycenter,dtw_barycenter_averaging,euclidean_barycenter
import xarray as xr

import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl


import goatools
import goatools.base
from goatools.base import download_go_basic_obo

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.semantic import semantic_similarity
from goatools.semantic import TermCounts, get_info_content

from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.go_enrichment import GOEnrichmentStudy

import holoviews as hv
hv.extension('bokeh')
random.seed(42)
np.random.seed(42)

warnings.filterwarnings(action='once',category=UserWarning)

In [ ]:
def parallel_norm_soft_dtw(X,chunk_size=200):
    X_dask = da.from_array(X,chunks=(chunk_size,X.shape[1],X.shape[2]))
    soft_dtw_arr = da.blockwise(cdist_soft_dtw,"ik",X_dask,"itd",X_dask,"ktd",concatenate=True).compute()
    d_ii = np.diag(soft_dtw_arr)
    norm_soft_dtw_arr = soft_dtw_arr - (.5 * (d_ii.reshape((-1, 1)) + d_ii.reshape((1, -1))))
    return norm_soft_dtw_arr

def norm_soft_dtw(X):
    soft_dtw_arr = cdist_soft_dtw(X)
    d_ii = np.diag(soft_dtw_arr)
    norm_soft_dtw_arr = soft_dtw_arr - (.5 * (d_ii.reshape((-1, 1)) + d_ii.reshape((1, -1))))
    return norm_soft_dtw_arr

def inter_exp_distances(x_chunk,exp_ids=[1,2]):
    exp_chunk_arr = []
    mean_exp_dist_dict = {}
    
    exp_chunk_lens = [len(x_chunk[x_chunk['Experiment #']==i]) for i in exp_ids]
    
    if np.all([chunk_len>1 for chunk_len in exp_chunk_lens]):

        for i in exp_ids:
            exp_chunk_arr.append(np.swapaxes(np.stack(x_chunk[x_chunk['Experiment #']==i]['Feature Vector'].values),1,2))

        exp_chunk_lens = [len(exp_chunk) for exp_chunk in exp_chunk_arr]
        exp_chunk_intervals = np.concatenate([np.array([0]),np.add.accumulate(exp_chunk_lens)])

        if np.all([chunk_len>1 for chunk_len in exp_chunk_lens]):
            exp_chunk_arr = np.concatenate(exp_chunk_arr)

            dist_out = norm_soft_dtw(exp_chunk_arr)

            for i in range(len(exp_ids)):
                for j in range(i,len(exp_ids)):
                    dist_section = dist_out[exp_chunk_intervals[i]:exp_chunk_intervals[i+1],exp_chunk_intervals[j]:exp_chunk_intervals[j+1]]
                    if i == j:
                        dist_section = np.trim_zeros(np.sort(np.triu(dist_section).flatten()))
                    else:
                        dist_section = dist_section.flatten()
                    mean_exp_dist_dict[(exp_ids[i],exp_ids[j])] = np.mean(dist_section)

    else:
        for i in range(len(exp_ids)):
            for j in range(i,len(exp_ids)):
                mean_exp_dist_dict[(exp_ids[i],exp_ids[j])] = np.NaN
    
    return mean_exp_dist_dict


def search_go(ns2assoc, obodag, inv_gene_to_id, go_term):
    namespace_abbv = {"biological_process":"BP","molecular_function":"MF","cellular_component":"CC"}
    
    print("Searching for " + str(obodag[go_term].name))
    namespace = namespace_abbv[obodag[go_term].namespace]
    child_goterms = list(obodag[go_term].get_all_children())
    gene_list = [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if go_term in val]
    for child_goterm in child_goterms:    
        gene_list += [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if child_goterm in val]
    gene_list = sorted(list(set(gene_list)))
    return gene_list

def selection_fn(item, gene_name):
    is_gene = item["Gene"]==gene_name
    if is_gene:
        return item["TargetID"]
    else:
        return 0

def highlight_gene_group(an_df,selection_list):

    highlight_genes_df = copy.deepcopy(an_df)
    
    selection_list = sorted(list(set(highlight_genes_df.obs["Gene"].unique().tolist())&set(selection_list)))

    for i,selected_gene in enumerate(selection_list):
        selected_series = (highlight_genes_df.obs["Gene"]==selected_gene).astype("category")
        selected_series = selected_series.cat.reorder_categories([True,False])
        highlight_genes_df.obs["Selected Genes: " + str(i)] = selected_series
            
    selected_series = (highlight_genes_df.obs["Gene"].isin(selection_list)).astype("category")
    selected_series = selected_series.cat.reorder_categories([True,False])
    highlight_genes_df.obs["All Genes"] = selected_series

    # selected_series = (paga_df.obs["Gene"]=="ftsZ").astype(float)
    # selected_series[selected_series==0.] = np.NaN
    # paga_df.obs["Selected Genes"] = selected_series

    fig = sc.pl.umap(highlight_genes_df, title=selection_list + ["All Genes"], color=["Selected Genes: " + str(i) for i in range(len(selection_list))] + ["All Genes"],
                     groups=[True], show=False, legend_loc='right margin', add_outline=False, size=50, return_fig=True,
                     palette = {True:"red",False:"lightgrey"})# palette ={}

    return fig

def highlight_sgrnas(an_df,selection_list):

    highlight_genes_df = copy.deepcopy(an_df)
    highlight_genes_df.obs["tempindex"] = highlight_genes_df.obs.index
    
    selection_list = sorted(list(set(highlight_genes_df.obs["tempindex"].unique().tolist())&set(selection_list)))        
    selected_series = (highlight_genes_df.obs["tempindex"].isin(selection_list)).astype("category")
    selected_series = selected_series.cat.reorder_categories([True,False])
    highlight_genes_df.obs["All sgRNAs"] = selected_series

    # selected_series = (paga_df.obs["Gene"]=="ftsZ").astype(float)
    # selected_series[selected_series==0.] = np.NaN
    # paga_df.obs["Selected Genes"] = selected_series

    fig = sc.pl.umap(highlight_genes_df, title="All sgRNAs", color="All sgRNAs",
                     groups=[True], show=False, legend_loc='right margin', add_outline=False, size=50, return_fig=True,
                     palette = {True:"red",False:"lightgrey"})# palette ={}

    return fig

### Initial Data Processing

Here, I am going to try and replicate (to some extant) the corrections from "Genomewide phenotypic analysis of growth, cell morphogenesis, and cell cycle events in Escherichia coli"

#### Start Dask

In [ ]:
dask_controller = tr.dask_controller(
    walltime="02:00:00",
    local=False,
    n_workers=50,
    n_workers_min=50,
    memory="8GB",
    working_directory="/home/lag36/scratch/lag36/dask/",
)
dask_controller.startdask()

# #small testing deployment

# # dask_controller = tr.trcluster.dask_controller(
# #     walltime="02:00:00",
# #     local=False,
# #     n_workers=40,
# #     n_workers_min=40,
# #     memory="16GB",
# #     working_directory="/home/de64/scratch/de64/dask",
# # )
# # dask_controller.startdask()

In [ ]:
dask_controller.daskclient

### Standardizing data before clustering

In [ ]:
sgRNA_df_merged = pd.read_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-06_sgRNA_Timeseries_df.pkl")

ss_mean_cv_df = pd.read_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Steady_State_MeanCV.pkl")

lDE23_nanopore = pd.read_csv("/home/lag36/gutiloluis/2023-01-11_lDE25-Run3/lDE24_final_nanopore_df.tsv",sep="\t")

lDE23_design = pd.read_csv("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2021-04-20_Essential_KO_Lib_df_coords_corrected.tsv",sep="\t")
# lDE23_design = pd.read_csv("/home/de64/group/nanopore/dev_notebooks/2022-06-09_snakemake_lDE23/2021-04-20_Essential_KO_Lib_df_coords_corrected.tsv",sep="\t")

ss_mean_cv_df['Mean: Delta time (hr)'] = ss_mean_cv_df['Mean: Delta time (s)']/3600
ss_mean_cv_df['Variance: Delta time (hr)'] = ss_mean_cv_df['Variance: Delta time (s)']/(3600**2)
control_ss_mean_cv_df = ss_mean_cv_df[ss_mean_cv_df["Category"].isin(['OnlyPlasmid', 'NoTarget'])]
sgRNA_df_merged['Kernel Trace: Delta time (hr)'] = sgRNA_df_merged['Kernel Trace: Delta time (s)']/3600

In [ ]:
# HERE >>>
sem_thr = 0.5

sem_params = ["Delta time (s)","Septum Displacement Length Normalized","major_axis_length",\
              "minor_axis_length","mCherry mean_intensity","Instantaneous Growth Rate: Volume"]

for param in sem_params:
    sgRNA_df_merged["Mean: SEM: Kernel Trace: " + param + ": Transformed: z score"] = sgRNA_df_merged["SEM: Kernel Trace: " + param + ": Transformed: z score"].apply(lambda x: np.mean(x))

sgRNA_df_merged["Below SEM Threshold"] = sgRNA_df_merged["SEM: Feature Vector"].apply(lambda x: np.all(x<sem_thr))
genewise_retention_rate = sgRNA_df_merged.groupby("Gene")["Below SEM Threshold"].apply(lambda x: np.sum(x)/len(x))

sgRNA_df_merged = sgRNA_df_merged[sgRNA_df_merged["Below SEM Threshold"]]
ss_mean_cv_df = ss_mean_cv_df.loc[sgRNA_df_merged.index.tolist()].copy()

## Processing and Filters for Clustering

In [ ]:
# zscore_labels = ['Delta time (hr)','Septum Displacement Length Normalized','major_axis_length',\
#           'minor_axis_length','mCherry mean_intensity','Instantaneous Growth Rate: Volume']

zscore_labels = ['Delta time (hr)',
               'Septum Displacement Length Normalized',
               'major_axis_length',
               'minor_axis_length',
               'mCherry mean_intensity',
               "Number of Nucleoids",
               'YFP mean_intensity',
               "Total Nucleoid Area",
               "Mean Nucleoid Area",
               "Nucleoid Mean Intensity",
               "Nucleoid Major Axis Length",
               "Nucleoid Minor Axis Length",
               'Instantaneous Growth Rate: Volume']

# note nucleoid not included at this point
sgRNA_df_merged = sgRNA_df_merged[sgRNA_df_merged["Feature Vector"].apply(lambda x: ~np.any(np.isnan(x)))]
sgRNA_df_merged = sgRNA_df_merged[sgRNA_df_merged["Category"].isin(["Target","NonEssential"])]

zero_vector = np.zeros((1,sgRNA_df_merged["Feature Vector"].iloc[0].shape[0]))
feature_arr = np.array(sgRNA_df_merged["Feature Vector"].tolist())
feature_arr_abs = np.abs(feature_arr)

## Getting Scores for parameter overlays
sgRNA_df_merged["Integrated Z-scores"] = [item/feature_arr.shape[2] for item in sp.integrate.simpson(feature_arr,axis=2)]
sgRNA_df_merged["Integrated Absolute Z-scores"] = [item for item in sp.integrate.simpson(feature_arr_abs,axis=2)]
sgRNA_df_merged["Integrated Absolute Z-score Max"] = sgRNA_df_merged["Integrated Absolute Z-scores"].apply(lambda x: np.max(x))

sgRNA_df_merged["Last Timepoint Z-scores"] = [item for item in feature_arr[:,:,-1]]
sgRNA_df_merged["Feature Max Absolute Z-score"] = [item for item in np.max(feature_arr_abs,axis=2)]
sgRNA_df_merged["Feature 90th Percentile Absolute Z-score"] = [item for item in np.percentile(feature_arr_abs,90,axis=2)]
sgRNA_df_merged["Feature Max Magnitude Z-score"] = [np.diag(feature_arr[i][:,item]) for i,item in enumerate(np.argmax(feature_arr_abs,axis=2))]
for i, label in enumerate(zscore_labels):
    sgRNA_df_merged[label + ": Integrated Z-score"] = sgRNA_df_merged["Integrated Z-scores"].apply(lambda x: x[i])
    sgRNA_df_merged[label + ": Last Timepoint Z-score"] = sgRNA_df_merged["Last Timepoint Z-scores"].apply(lambda x: x[i])
    sgRNA_df_merged[label + ": Max Absolute Z-score"] = sgRNA_df_merged["Feature Max Absolute Z-score"].apply(lambda x: x[i])
    sgRNA_df_merged[label + ": 90th Percentile Absolute Z-score"] = sgRNA_df_merged["Feature 90th Percentile Absolute Z-score"].apply(lambda x: x[i])
    sgRNA_df_merged[label + ": Max Magnitude Z-score"] = sgRNA_df_merged["Feature Max Magnitude Z-score"].apply(lambda x: x[i])
    sgRNA_df_merged[label + ": Z-score Timeseries"] = sgRNA_df_merged["Feature Vector"].apply(lambda x: x[i])
    sgRNA_df_merged[label + ": Max Magnitude Minus Last Timepoint Z-score"] = sgRNA_df_merged[label + ": Max Magnitude Z-score"]-sgRNA_df_merged[label + ": Last Timepoint Z-score"]

sgRNA_df_merged["Feature Max Absolute Component"] = sgRNA_df_merged["Feature Max Absolute Z-score"].apply(lambda x: np.max(x))
sgRNA_df_merged["Feature Max Absolute 90th Percentile Component"] = sgRNA_df_merged["Feature 90th Percentile Absolute Z-score"].apply(lambda x: np.max(x))

### Setting Max Z-score Threshold

In [ ]:
z_score_type = ["Integrated","90th Percentile","Max"][2]
strong_effect_threshold = [0,1.5,2.5][0]
n_neighbors = [5,10,15][1]

# rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-01-02_lDE23_32Hour_Analysis"
rootpath = "/home/lag36/gutiloluis/2023-01-11_lDE25-Run3/"
z_score_thr_dir = rootpath + "/Z_Score_Thr_" + str(strong_effect_threshold) + "_N_Neighbors_" + str(n_neighbors)
clustree_path = z_score_thr_dir + "/Clustree"
umap_path = z_score_thr_dir + "/UMAP"
compare_path = z_score_thr_dir + "/Comparison"

tr.writedir(z_score_thr_dir, overwrite=False)
tr.writedir(clustree_path, overwrite=False)
tr.writedir(umap_path, overwrite=False)

if z_score_type == "Integrated":
    z_score_integrated_series = sgRNA_df_merged["Integrated Absolute Z-score Max"]
    min_v,max_v = np.min(z_score_integrated_series),np.percentile(z_score_integrated_series,99)
    sgRNA_df_merged_filtered = sgRNA_df_merged[z_score_integrated_series>=strong_effect_threshold]

    plt.figure(figsize=(8,8))
    plt.title("Integrated Feature Max")
    plt.hist(sgRNA_df_merged[z_score_integrated_series<strong_effect_threshold]["Integrated Absolute Z-score Max"].tolist(),bins=50,range=(min_v,max_v))
    plt.hist(sgRNA_df_merged[z_score_integrated_series>=strong_effect_threshold]["Integrated Absolute Z-score Max"].tolist(),bins=50,range=(min_v,max_v))
    plt.show()

    #include dropped genes
    unique_genes, gene_counts = np.unique(sgRNA_df_merged_filtered["Gene"][sgRNA_df_merged_filtered["Gene"].apply(lambda x: type(x)==str)].tolist(), return_counts=True)
    plt.title("sgRNAs per Gene")
    plt.xticks(range(0,20,2),labels=range(0,20,2))
    plt.hist(gene_counts,bins=np.arange(20)-0.5)
    plt.show()

elif z_score_type == "Max":
    ##current max
    z_score_max_series = sgRNA_df_merged["Feature Max Absolute Component"]
    min_v,max_v = np.min(z_score_max_series),np.percentile(z_score_max_series,99)
    sgRNA_df_merged_filtered = sgRNA_df_merged[z_score_max_series>=strong_effect_threshold]

    plt.figure(figsize=(8,8))
    plt.title("Feature Max Absolute Component")
    plt.hist(sgRNA_df_merged[z_score_max_series<strong_effect_threshold]["Feature Max Absolute Component"].tolist(),bins=50,range=(min_v,max_v))
    plt.hist(sgRNA_df_merged[z_score_max_series>=strong_effect_threshold]["Feature Max Absolute Component"].tolist(),bins=50,range=(min_v,max_v))
    plt.show()

    #include dropped genes
    unique_genes, gene_counts = np.unique(sgRNA_df_merged_filtered["Gene"][sgRNA_df_merged_filtered["Gene"].apply(lambda x: type(x)==str)].tolist(), return_counts=True)
    plt.title("sgRNAs per Gene")
    plt.xticks(range(0,20,2),labels=range(0,20,2))
    plt.hist(gene_counts,bins=np.arange(20)-0.5)
    plt.show()

elif z_score_type=="90th Percentile":
    ##current max
    z_score_perc_series = sgRNA_df_merged["Feature Max Absolute 90th Percentile Component"]
    min_v,max_v = np.min(z_score_perc_series),np.percentile(z_score_perc_series,99)
    sgRNA_df_merged_filtered = sgRNA_df_merged[z_score_perc_series>=strong_effect_threshold]

    plt.figure(figsize=(8,8))
    plt.title("Feature Max Absolute Component")
    plt.hist(sgRNA_df_merged[z_score_perc_series<strong_effect_threshold]["Feature Max Absolute 90th Percentile Component"].tolist(),bins=50,range=(min_v,max_v))
    plt.hist(sgRNA_df_merged[z_score_perc_series>=strong_effect_threshold]["Feature Max Absolute 90th Percentile Component"].tolist(),bins=50,range=(min_v,max_v))
    plt.show()

    #include dropped genes
    unique_genes, gene_counts = np.unique(sgRNA_df_merged_filtered["Gene"][sgRNA_df_merged_filtered["Gene"].apply(lambda x: type(x)==str)].tolist(), return_counts=True)
    plt.title("sgRNAs per Gene")
    plt.xticks(range(0,20,2),labels=range(0,20,2))
    plt.hist(gene_counts,bins=np.arange(20)-0.5)
    plt.show()


### Checking sgRNA Sampling Distribution

In [ ]:
design_gene_list = np.unique(lDE23_design["Gene"][lDE23_design["Gene"].apply(lambda x: type(x)==str)].tolist()).tolist()
design_gene_names,design_count = np.unique(lDE23_design["Gene"][lDE23_design["Gene"].apply(lambda x: type(x)==str)].tolist(), return_counts=True)
design_gene_count_dict = dict(zip(design_gene_names,design_count))
design_gene_count_dict = {gene:(design_gene_count_dict[gene] if gene in design_gene_count_dict.keys() else 0) for gene in design_gene_list}
design_gene_count_list = list(design_gene_count_dict.values())

lDE23_nanopore_sgrna_only = lDE23_nanopore.groupby("oDEPool7_id").apply(lambda x: x.iloc[0])
nanopore_gene_names,nanopore_count = np.unique(lDE23_nanopore_sgrna_only["Gene"][lDE23_nanopore_sgrna_only["Gene"].apply(lambda x: type(x)==str)].tolist(), return_counts=True)
nanopore_gene_count_dict = dict(zip(nanopore_gene_names,nanopore_count))
nanopore_gene_count_dict = {gene:(nanopore_gene_count_dict[gene] if gene in nanopore_gene_count_dict.keys() else 0) for gene in design_gene_list}
nanopore_gene_count_list = list(nanopore_gene_count_dict.values())

experiment_gene_names,experiment_count = np.unique(sgRNA_df_merged["Gene"][sgRNA_df_merged["Gene"].apply(lambda x: type(x)==str)].tolist(), return_counts=True)
experiment_gene_count_dict = dict(zip(experiment_gene_names,experiment_count))
experiment_gene_count_dict = {gene:(experiment_gene_count_dict[gene] if gene in experiment_gene_count_dict.keys() else 0) for gene in design_gene_list}
experiment_gene_count_list = list(experiment_gene_count_dict.values())

significant_gene_names,significant_count = np.unique(sgRNA_df_merged_filtered["Gene"][sgRNA_df_merged_filtered["Gene"].apply(lambda x: type(x)==str)].tolist(), return_counts=True)
significant_gene_count_dict = dict(zip(significant_gene_names,significant_count))
significant_gene_count_dict = {gene:(significant_gene_count_dict[gene] if gene in significant_gene_count_dict.keys() else 0) for gene in design_gene_list}
significant_gene_count_list = list(significant_gene_count_dict.values())

In [ ]:
fig = plt.figure(figsize=(14,10))
plt.subplot(2, 2, 1)
plt.title("Library Design",fontsize=20)
plt.xticks(range(0,70,10),labels=range(0,70,10))
plt.xlabel("# sgRNA per gene",fontsize=16)
plt.xticks(fontsize=16)
plt.ylabel("# genes",fontsize=16)
plt.yticks(fontsize=16)
plt.hist(design_gene_count_list,bins=np.arange(70)-0.5)

plt.subplot(2, 2, 2)
plt.title("Sequenced Library",fontsize=20)
plt.xticks(range(0,70,10),labels=range(0,70,10))
plt.xlabel("# sgRNA per gene",fontsize=16)
plt.xticks(fontsize=16)
plt.ylabel("# genes",fontsize=16)
plt.yticks(fontsize=16)
plt.hist(nanopore_gene_count_list,bins=np.arange(70)-0.5)

plt.subplot(2, 2, 3)
plt.title("Observed in Experiment ($\geq$ 10 Lineages)",fontsize=20)
plt.xticks(range(0,70,10),labels=range(0,70,10))
plt.xlabel("# sgRNA per gene",fontsize=16)
plt.xticks(fontsize=16)
plt.ylabel("# genes",fontsize=16)
plt.yticks(fontsize=16)
plt.hist(experiment_gene_count_list,bins=np.arange(70)-0.5)

if z_score_type == "Integrated":
    plt.subplot(2, 2, 4)
    plt.title("Significant Z-score ($max_{i}(\int_{0}^{t_{f}} Z_{i}(t) dt) \geq " + str(strong_effect_threshold) + "$)",fontsize=20)
    plt.xticks(range(0,70,10),labels=range(0,70,10))
    plt.xlabel("# sgRNA per gene",fontsize=16)
    plt.xticks(fontsize=16)
    plt.ylabel("# genes",fontsize=16)
    plt.yticks(fontsize=16)
    plt.hist(significant_gene_count_list,bins=np.arange(70)-0.5)

elif z_score_type == "Max":
    plt.subplot(2, 2, 4)
    plt.title("Significant Z-score ($max_{i,t}(Z_{i}(t)) \geq " + str(strong_effect_threshold) + "$)",fontsize=20)
    plt.xticks(range(0,70,10),labels=range(0,70,10))
    plt.xlabel("# sgRNA per gene",fontsize=16)
    plt.xticks(fontsize=16)
    plt.ylabel("# genes",fontsize=16)
    plt.yticks(fontsize=16)
    plt.hist(significant_gene_count_list,bins=np.arange(70)-0.5)

elif z_score_type == "90th Percentile":
    plt.subplot(2, 2, 4)
    plt.title("Significant Z-score ($90thPerc_{i,t}(Z_{i}(t)) \geq " + str(strong_effect_threshold) + "$)",fontsize=20)
    plt.xticks(range(0,70,10),labels=range(0,70,10))
    plt.xlabel("# sgRNA per gene",fontsize=16)
    plt.xticks(fontsize=16)
    plt.ylabel("# genes",fontsize=16)
    plt.yticks(fontsize=16)
    plt.hist(significant_gene_count_list,bins=np.arange(70)-0.5)

plt.tight_layout()
fig.savefig(z_score_thr_dir + "/sgRNA_Frequencies.png",dpi=300)

In [ ]:
sgRNA_df_merged_filtered = sgRNA_df_merged_filtered.join(ss_mean_cv_df.drop(['sgRNA', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID','N Observations'],axis=1),on="oDEPool7_id")
sgRNA_df_merged_filtered['N Match'] = 20. - sgRNA_df_merged_filtered['N Mismatch']
del_N_match_series = sgRNA_df_merged_filtered.groupby("TargetID").apply(lambda x: x['N Match']-np.min(x['N Match']))
del_N_match_series = del_N_match_series.droplevel("TargetID")
sgRNA_df_merged_filtered["Delta N Match"] = del_N_match_series

### Computing Distance Matrix

In [ ]:
### soft-DTW Calculation

X = np.array(sgRNA_df_merged_filtered["Feature Vector"].tolist())
X = np.swapaxes(X,1,2)
norm_soft_dtw_arr = parallel_norm_soft_dtw(X)

## Leiden Analysis

### Scanning Leiden Parameters

In [ ]:
### Initialize Anndata Object
an_df_merged = anndata.AnnData(X=X.reshape(X.shape[0],-1),obs=sgRNA_df_merged_filtered) # AnnData container to use scanpy functions with unwrapped time vector

### Compute KNN Graph
n_pcs = 20 #This shouldn't affect anything

sc.pp.neighbors(an_df_merged, n_neighbors=n_neighbors, n_pcs=n_pcs)
knn_indices, knn_dists, forest = sc.neighbors.compute_neighbors_umap(norm_soft_dtw_arr, n_neighbors=n_neighbors, metric='precomputed' )
an_df_merged.uns["neighbors"]["distances"], an_df_merged.uns["neighbors"]["connectivities"] = sc.neighbors._compute_connectivities_umap(
    knn_indices,
    knn_dists,
    an_df_merged.shape[0],
    n_neighbors, # change to neighbors you plan to use
)
an_df_merged.obsp["distances"] = an_df_merged.uns["neighbors"]["distances"]
an_df_merged.obsp["connectivities"] = an_df_merged.uns["neighbors"]["connectivities"]
an_df_merged.obsp["soft_dtw"] = norm_soft_dtw_arr

### Clustering Metrics

- Silhouette Score - internal evaluation
- Adjusted Rand Index - external evaluation
    - TargetIDs could be used gainfully here
- Adjusted Mutual Information

- Need to also consider what I am evaluating on. TargetIDs are probably good for getting a range, but will be less useful in evaluating cluster quality later.

In [ ]:
def get_top_genes(x,n_top=2,n_gene_names_per_line=2):
    genes,counts = np.unique(x,return_counts=True)
    sorted_genes = genes[np.argsort(counts)[::-1]]
    sorted_genes = list(sorted_genes[:n_top])
    sorted_genes = [".\n".join(gene.split()[:n_gene_names_per_line]) for gene in sorted_genes]
    return sorted_genes

def get_recall_df(an_df,minimum_total_count=2,recall_metric="TargetID",cluster_label="leiden"):
    an_df_copy = copy.deepcopy(an_df)
    recall_metric_count_dict = dict(zip(*np.unique(an_df_copy.obs[recall_metric],return_counts=True)))
    all_counts = np.sum(list(recall_metric_count_dict.values()))
    
    top_2_genes_series = an_df_copy.obs.groupby(cluster_label).apply(lambda x: "\n".join(get_top_genes(x["Gene"])))
    top_2_genes_df = pd.DataFrame(top_2_genes_series).rename(columns={0:"Top_2_Genes"})
    top_2_genes_df_columns = top_2_genes_df.columns
    an_df_addition = an_df_copy.obs[[cluster_label]].reset_index().merge(top_2_genes_df, on=cluster_label).set_index('oDEPool7_id')
    an_df_copy.obs[top_2_genes_df_columns] = an_df_addition[top_2_genes_df_columns]
    
    recall_count_in_cluster = an_df_copy.obs.groupby(cluster_label).apply(lambda x: dict(zip(*np.unique(x[recall_metric],return_counts=True))))
    recall_count_all_clusters = an_df_copy.obs.groupby(cluster_label).apply(lambda x: {key:recall_metric_count_dict[key] for key in np.unique(x[recall_metric])})
    recall_df = pd.concat([recall_count_in_cluster,recall_count_all_clusters],axis=1).rename(columns={0:"In Cluster",1:"All Clusters"})

    recall_df[recall_metric + "_Recall_List_" + cluster_label] = recall_df.apply(lambda x: [x["In Cluster"][key]/x["All Clusters"][key] for key in x["In Cluster"].keys() if x["All Clusters"][key]>=minimum_total_count], axis=1)
    recall_df = recall_df[recall_df[recall_metric + "_Recall_List_" + cluster_label].apply(len)>0]
    
    recall_df[recall_metric + "_Mean_Recall_" + cluster_label] = recall_df[recall_metric + "_Recall_List_" + cluster_label].apply(lambda x: np.mean(x))
    recall_df[recall_metric + "_Median_Recall_" + cluster_label] = recall_df[recall_metric + "_Recall_List_" + cluster_label].apply(lambda x: np.median(x))
    recall_df[recall_metric + "_Max_Recall_" + cluster_label] = recall_df[recall_metric + "_Recall_List_" + cluster_label].apply(lambda x: np.max(x))
    recall_df[recall_metric + "_Min_Recall_" + cluster_label] = recall_df[recall_metric + "_Recall_List_" + cluster_label].apply(lambda x: np.min(x))
    recall_df[recall_metric + "_Top2_Recall_" + cluster_label] = recall_df[recall_metric + "_Recall_List_" + cluster_label].apply(lambda x: sorted(x)[-2] if len(x)>1 else 0.)
    recall_df = recall_df.drop(columns=["In Cluster","All Clusters"])
    
    return recall_df

In [ ]:
max_val = 100
min_val = 0.04
nsteps = 50

resolution_list = np.logspace(np.log10(min_val),np.log10(max_val),num=nsteps).tolist()

unique_targetids,targetid_counts = np.unique(an_df_merged.obs["TargetID"],return_counts=True)
non_singleton_targetids = unique_targetids[targetid_counts>1]

unique_genes,gene_counts = np.unique(an_df_merged.obs["Gene"],return_counts=True)
non_singleton_genes = unique_genes[gene_counts>1]

sil_score_list = []
targetid_rand_score_list = []
targetid_mut_info_score_list = []
targetid_homogeneity_list = []
targetid_completeness_list = []
targetid_v_measure_list = []

all_recall_target = []
mean_cluster_recall_target = []
max_cluster_recall_target = []
all_recall_gene = []
mean_cluster_recall_gene = []
max_cluster_recall_gene = []

gene_rand_score_list = []
gene_mut_info_score_list = []

for resolution in resolution_list:
    sc.tl.leiden(an_df_merged, resolution=resolution, n_iterations=-1)
    leiden_labels = an_df_merged.obs['leiden'].astype(int).tolist()
    targetid_labels = an_df_merged.obs['TargetID'].astype(int).tolist()
    
    an_df_merged_no_singleton = an_df_merged[an_df_merged.obs["TargetID"].isin(non_singleton_targetids)]
    leiden_labels_no_singleton = an_df_merged_no_singleton.obs["leiden"].astype(int).tolist()
    targetid_labels_no_singleton = an_df_merged_no_singleton.obs["TargetID"].astype(int).tolist()
    
    sil_score = skl.metrics.silhouette_score(norm_soft_dtw_arr, leiden_labels, metric='precomputed')
    sil_score_list.append(sil_score)
    
    targetid_rand_score = skl.metrics.adjusted_rand_score(targetid_labels_no_singleton, leiden_labels_no_singleton)
    targetid_mut_info_score = skl.metrics.adjusted_mutual_info_score(targetid_labels_no_singleton, leiden_labels_no_singleton)
    
    targetid_rand_score_list.append(targetid_rand_score)
    targetid_mut_info_score_list.append(targetid_mut_info_score)
    
    target_recall_df = get_recall_df(an_df_merged)
    gene_recall_df = get_recall_df(an_df_merged,recall_metric="Gene")
    
    all_recall_list_target = [val for lst in target_recall_df["TargetID_Recall_List_leiden"].tolist() for val in lst]
    mean_cluster_recall_list_target = target_recall_df["TargetID_Mean_Recall_leiden"].tolist()
    max_cluster_recall_list_target = target_recall_df["TargetID_Max_Recall_leiden"].tolist()
    
    all_recall_list_gene = [val for lst in gene_recall_df["Gene_Recall_List_leiden"].tolist() for val in lst]
    mean_cluster_recall_list_gene = gene_recall_df["Gene_Mean_Recall_leiden"].tolist()
    max_cluster_recall_list_gene = gene_recall_df["Gene_Max_Recall_leiden"].tolist()
    
    all_recall_target.append(all_recall_list_target)
    mean_cluster_recall_target.append(mean_cluster_recall_list_target)
    max_cluster_recall_target.append(max_cluster_recall_list_target)
    
    all_recall_gene.append(all_recall_list_gene)
    mean_cluster_recall_gene.append(mean_cluster_recall_list_gene)
    max_cluster_recall_gene.append(max_cluster_recall_list_gene)
    

    an_df_merged_no_singleton_genes = an_df_merged[an_df_merged.obs["Gene"].isin(non_singleton_genes)]
    leiden_labels_no_singleton_genes = an_df_merged_no_singleton_genes.obs["leiden"].astype(int).tolist()
    gene_labels_no_singleton_genes = an_df_merged_no_singleton_genes.obs["Gene"].astype(str).tolist()
    
    gene_rand_score = skl.metrics.adjusted_rand_score(gene_labels_no_singleton_genes, leiden_labels_no_singleton_genes)
    gene_mut_info_score = skl.metrics.adjusted_mutual_info_score(gene_labels_no_singleton_genes, leiden_labels_no_singleton_genes)
    gene_rand_score_list.append(gene_rand_score)
    gene_mut_info_score_list.append(gene_mut_info_score)

# Plotting Stuff

#### Plotting Recall vs Resolution

In [ ]:
output_target_recall_df = pd.DataFrame([[resolution_list[i],recall] for i,item in enumerate(all_recall_target) for recall in item],columns=["Resolution","Recall"])
output_gene_recall_df = pd.DataFrame([[resolution_list[i],recall] for i,item in enumerate(all_recall_gene) for recall in item],columns=["Resolution","Recall"])

output_target_mean_clster_recall_df = pd.DataFrame([[resolution_list[i],recall] for i,item in enumerate(mean_cluster_recall_target) for recall in item],columns=["Resolution","Recall"])
output_gene_mean_clster_recall_df = pd.DataFrame([[resolution_list[i],recall] for i,item in enumerate(mean_cluster_recall_gene) for recall in item],columns=["Resolution","Recall"])

output_target_max_clster_recall_df = pd.DataFrame([[resolution_list[i],recall] for i,item in enumerate(max_cluster_recall_target) for recall in item],columns=["Resolution","Recall"])
output_gene_max_clster_recall_df = pd.DataFrame([[resolution_list[i],recall] for i,item in enumerate(max_cluster_recall_gene) for recall in item],columns=["Resolution","Recall"])

In [ ]:
fig = plt.figure()
# sns.lineplot(x="Resolution",y="Recall",data=output_target_recall_df, ci="sd",label="TargetID")
sns.lineplot(x="Resolution",y="Recall",data=output_gene_recall_df, ci="sd",label="Gene")
plt.ylim(-0.1,1.2)
plt.xscale("log")
plt.xlabel("Resolution",fontsize=14)
plt.ylabel("Recall",fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(z_score_thr_dir + "/Recall_Vs_Resolution_Supplement.jpg",dpi=300)

In [ ]:
fig = plt.figure()
# sns.lineplot(x="Resolution",y="Recall",data=output_target_mean_clster_recall_df, ci="sd",label="TargetID")
sns.lineplot(x="Resolution",y="Recall",data=output_gene_mean_clster_recall_df, ci="sd",label="Gene")
plt.ylim(-0.1,1.2)
plt.xscale("log")
plt.xlabel("Resolution",fontsize=14)
plt.ylabel("Mean Cluster Recall",fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(z_score_thr_dir + "/Mean_Cluster_Recall_Vs_Resolution_Supplement.jpg",dpi=300)

In [ ]:
fig = plt.figure()
# sns.lineplot(x="Resolution",y="Recall",data=output_target_max_clster_recall_df, ci="sd",label="TargetID")
sns.lineplot(x="Resolution",y="Recall",data=output_gene_max_clster_recall_df, ci="sd",label="Gene")
plt.ylim(-0.1,1.2)
plt.xscale("log")
plt.xlabel("Resolution",fontsize=14)
plt.ylabel("Max Cluster Recall",fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(z_score_thr_dir + "/Max_Cluster_Recall_Vs_Resolution_Supplement.jpg",dpi=300)

#### Silhouette, Rand, and AMI vs Resolution (TargetID and Gene)

In [ ]:
fig = plt.figure()
plt.plot(resolution_list,sil_score_list)
plt.xscale("log")
plt.xlabel("Resolution",fontsize=14)
plt.ylabel("Silhouette Score",fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(z_score_thr_dir + "/Silhouette_Score_Vs_Resolution_Supplement.jpg",dpi=300)

In [ ]:
fig = plt.figure()
# plt.plot(resolution_list,targetid_rand_score_list,label="TargetID")
plt.plot(resolution_list,gene_rand_score_list,label="Gene")
plt.xscale("log")
plt.xlabel("Resolution",fontsize=14)
plt.ylabel("Adjusted Rand Index",fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig(z_score_thr_dir + "/ARI_Vs_Resolution_Supplement.jpg",dpi=300)

In [ ]:
fig = plt.figure()
# plt.plot(resolution_list,targetid_mut_info_score_list,label="TargetID")
plt.plot(resolution_list,gene_mut_info_score_list,label="Gene")
plt.xscale("log")
plt.xlabel("Resolution",fontsize=14)
plt.ylabel("Adjusted Mutual Information",fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig(z_score_thr_dir + "/AMI_Vs_Resolution_Supplement.jpg",dpi=300)

# Finished plots. Continue with NB

#### Setting Resolution Parameter Range and UMAP Projection

In [ ]:
min_dist = 0.1
spread = 5.

an_df_clustree = copy.deepcopy(an_df_merged)
an_df_clustree.obs = an_df_clustree.obs.drop(labels=["leiden"],axis=1)

sil_score_list = []
resolution_list = [0.2,0.5,1.,2.,4.,10.]
for i,resolution in enumerate(resolution_list):
    sc.tl.leiden(an_df_clustree, resolution=resolution, n_iterations=-1, key_added="L" + str(i))
    leiden_labels = an_df_clustree.obs["L" + str(i)].astype(int).tolist()
    targetid_labels = an_df_clustree.obs['TargetID'].astype(int).tolist()
    
    target_recall_df = get_recall_df(an_df_clustree,cluster_label="L" + str(i))
    gene_recall_df = get_recall_df(an_df_clustree,recall_metric="Gene",cluster_label="L" + str(i))
    
    an_df_clustree.obs = an_df_clustree.obs.join(target_recall_df.drop(columns=["TargetID_Recall_List_L" + str(i)],axis=1),on="L" + str(i),how="left")
    an_df_clustree.obs = an_df_clustree.obs.join(gene_recall_df.drop(columns=["Gene_Recall_List_L" + str(i)],axis=1),on="L" + str(i),how="left")
    
## Add PAGA -> UMAP projection based on resolution of 1
sc.tl.paga(an_df_clustree, groups='L2')
sc.pl.paga(an_df_clustree, add_pos=True, show=False)
sc.tl.umap(an_df_clustree, init_pos='paga', min_dist = min_dist, spread=spread)

## UMAP Analysis

### UMAP: Leiden Clusters

In [ ]:
umap_path

In [ ]:
leiden_labels = ["L" + str(i) for i,res in enumerate(resolution_list)]
leiden_titles = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)]
fig = sc.pl.umap(an_df_clustree, color=leiden_labels, title=leiden_titles, show=False,
           legend_loc='on data', edges=False, add_outline=False, size=100, return_fig=True,palette=vega_20_scanpy,ncols=2)
axes = fig.get_axes()
for ax in axes:
    ax.set_title(ax.get_title(), fontsize=18)
    ax.set_ylabel(ax.get_ylabel(), fontsize=18)
    ax.set_xlabel(ax.get_xlabel(), fontsize=18)
fig.savefig(umap_path + "/UMAP_leiden_clustering.png",dpi=300)

In [ ]:
dask_controller.shutdown()

In [ ]:
type(an_df_clustree)

# More Quality checking plots

### UMAP: Recall Overlay

#### UMAP: Median Recall (TrenchID)

In [ ]:
cbar_label = "Cluster Median TargetID Recall"
all_labels = ['TargetID_Median_Recall_L' + str(i) for i in range(len(resolution_list))]
title_list = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)]
# fig = sc.pl.umap(paga_df_merged, color=labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, return_fig=True,vcenter=0.,vmin=-2.5,vmax=2.5,cmap='RdBu_r',wspace=0.25,ncols=2)
# fig = sc.pl.umap(paga_df_merged, color=all_labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, norm=norm_list, return_fig=True,cmap='RdBu_r',wspace=0.25,ncols=2)

fig = sc.pl.umap(an_df_clustree, color=all_labels, show=False, title=title_list,
           legend_loc='on data', add_outline=False, size=50, vmin=0., vmax=1., return_fig=True,cmap='Reds',wspace=0.25,ncols=2)

axes = fig.get_axes()
for i,ax in enumerate(axes):
    if i%2==1:
        #colorbar settings
        ax.set_ylabel(cbar_label,fontsize=16,labelpad=10)
    else:
        #plot settings
        ax.set_title(ax.get_title(), fontsize=18)
        ax.set_ylabel(ax.get_ylabel(), fontsize=18)
        ax.set_xlabel(ax.get_xlabel(), fontsize=18)
plt.tight_layout()
fig.savefig(umap_path + "/UMAP_Median_TargetID_Recall.png",dpi=300)

#### UMAP: Median Recall (Gene)

In [ ]:
cbar_label = "Cluster Median Gene Recall"
all_labels = ['Gene_Median_Recall_L' + str(i) for i in range(len(resolution_list))]
title_list = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)]
# fig = sc.pl.umap(paga_df_merged, color=labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, return_fig=True,vcenter=0.,vmin=-2.5,vmax=2.5,cmap='RdBu_r',wspace=0.25,ncols=2)
# fig = sc.pl.umap(paga_df_merged, color=all_labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, norm=norm_list, return_fig=True,cmap='RdBu_r',wspace=0.25,ncols=2)

fig = sc.pl.umap(an_df_clustree, color=all_labels, show=False, title=title_list,
           legend_loc='on data', add_outline=False, size=50, vmin=0., vmax=1., return_fig=True,cmap='Reds',wspace=0.25,ncols=2)

axes = fig.get_axes()
for i,ax in enumerate(axes):
    if i%2==1:
        #colorbar settings
        ax.set_ylabel(cbar_label,fontsize=16,labelpad=10)
    else:
        #plot settings
        ax.set_title(ax.get_title(), fontsize=18)
        ax.set_ylabel(ax.get_ylabel(), fontsize=18)
        ax.set_xlabel(ax.get_xlabel(), fontsize=18)
plt.tight_layout()
fig.savefig(umap_path + "/UMAP_Median_Gene_Recall.png",dpi=300)

#### UMAP: Max Recall (TargetID)

In [ ]:
cbar_label = "Cluster Max TargetID Recall"
all_labels = ['TargetID_Max_Recall_L' + str(i) for i in range(len(resolution_list))]
title_list = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)]
# fig = sc.pl.umap(paga_df_merged, color=labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, return_fig=True,vcenter=0.,vmin=-2.5,vmax=2.5,cmap='RdBu_r',wspace=0.25,ncols=2)
# fig = sc.pl.umap(paga_df_merged, color=all_labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, norm=norm_list, return_fig=True,cmap='RdBu_r',wspace=0.25,ncols=2)

fig = sc.pl.umap(an_df_clustree, color=all_labels, show=False, title=title_list,
           legend_loc='on data', add_outline=False, size=50, vmin=0., vmax=1., return_fig=True,cmap='Reds',wspace=0.25,ncols=2)

axes = fig.get_axes()
for i,ax in enumerate(axes):
    if i%2==1:
        #colorbar settings
        ax.set_ylabel(cbar_label,fontsize=16,labelpad=10)
    else:
        #plot settings
        ax.set_title(ax.get_title(), fontsize=18)
        ax.set_ylabel(ax.get_ylabel(), fontsize=18)
        ax.set_xlabel(ax.get_xlabel(), fontsize=18)
plt.tight_layout()
fig.savefig(umap_path + "/UMAP_Max_TargetID_Recall.png",dpi=300)

#### UMAP: Max Recall (Gene)

In [ ]:
cbar_label = "Cluster Max Gene Recall"
all_labels = ['Gene_Max_Recall_L' + str(i) for i in range(len(resolution_list))]
title_list = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)]
# fig = sc.pl.umap(paga_df_merged, color=labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, return_fig=True,vcenter=0.,vmin=-2.5,vmax=2.5,cmap='RdBu_r',wspace=0.25,ncols=2)
# fig = sc.pl.umap(paga_df_merged, color=all_labels, show=False,
#            legend_loc='on data', add_outline=False, size=50, norm=norm_list, return_fig=True,cmap='RdBu_r',wspace=0.25,ncols=2)

fig = sc.pl.umap(an_df_clustree, color=all_labels, show=False, title=title_list,
           legend_loc='on data', add_outline=False, size=50, vmin=0., vmax=1., return_fig=True,cmap='Reds',wspace=0.25,ncols=2)

axes = fig.get_axes()
for i,ax in enumerate(axes):
    if i%2==1:
        #colorbar settings
        ax.set_ylabel(cbar_label,fontsize=16,labelpad=10)
    else:
        #plot settings
        ax.set_title(ax.get_title(), fontsize=18)
        ax.set_ylabel(ax.get_ylabel(), fontsize=18)
        ax.set_xlabel(ax.get_xlabel(), fontsize=18)
plt.tight_layout()
fig.savefig(umap_path + "/UMAP_Max_Gene_Recall.png",dpi=300)

# End quality check plots. UMAP Parameter Overlay

In [ ]:
z_score_thr_dir+"/AnnDataFull.h5ad"

In [ ]:
an_df_clustree.write(z_score_thr_dir+"/AnnDataFull.h5ad")

### UMAP: Parameter Overlay

In [ ]:
import pandas as pd
import numpy as np
df_pre = pd.read_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Preinduction_MeanCV.pkl")
df_post = pd.read_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Steady_State_MeanCV.pkl")

all_labels = ['Mean: Delta time (s)', # Changed from h
          'Mean: Septum Displacement Length Normalized',
          'Mean: major_axis_length',
          'Mean: minor_axis_length',
          'Mean: mCherry mean_intensity',
          'Mean: Number of Nucleoids',
          'Mean: YFP mean_intensity',
          'Mean: Total Nucleoid Area',
          'Mean: Nucleoid Area Mean',
          'Mean: Nucleoid Mean Intensity Mean',
          'Mean: Nucleoid Major Axis Length Mean',
          'Mean: Nucleoid Minor Axis Length Mean', 
          'Mean: Instantaneous Growth Rate: Volume']



vmin = np.zeros(len(all_labels))
vcenter = np.zeros(len(all_labels))
vmax = np.zeros(len(all_labels))
for i, label in enumerate(all_labels):
    # print(label, label in df_post.columns)\
    vcenter[i] = df_pre[label].mean()
    vmin[i] = df_post[label].min()
    vmax[i] = df_post[label].max()
    # print("vcenter", df_pre[label].mean())
    
#Change to hours
vcenter[0] /= 3600 
vmin[0] /= 3600
vmax[0]/=3600

vcenter = list(vcenter)
vmin = list(vmin)
vmax = list(vmax)

all_labels[0] = 'Mean: Delta time (hr)'
title_list = all_labels

In [ ]:
# Version with hand-picked vcenter, vmin, vmax
# for label in all_labels:
    # vmin,vmax = np.nanpercentile(an_df_clustree.obs[label],1),np.nanpercentile(an_df_clustree.obs[label],99)
    # vcenter = median_vals_dict[label]
    # min_deviation,max_deviation = (vcenter-vmin,vmax-vcenter)
    # if min_deviation>max_deviation:
    #     vmax = vcenter + min_deviation
    # else:
    #     vmin = vcenter - max_deviation
    # min_vals.append(vmin)
    # max_vals.append(vmax)

fig = sc.pl.umap(an_df_clustree, color=all_labels, show=False,\
            title=title_list,legend_loc='on data', add_outline=False, size=50, vmin=vmin, vmax=vmax, \
                 return_fig=True,cmap='RdBu_r',wspace=0.25,ncols=2, vcenter=vcenter)

axes = fig.get_axes()
# for i,ax in enumerate(axes):
#     if i%2==1:
#         #colorbar settings
#         ax.set_ylabel(cbar_label,fontsize=16,labelpad=10)
#     else:
#         #plot settings
#         ax.set_title(ax.get_title(), fontsize=18)
#         ax.set_ylabel(ax.get_ylabel(), fontsize=18)
#         ax.set_xlabel(ax.get_xlabel(), fontsize=18)

fig.savefig(umap_path + "/UMAP_Steady_State_Means.png",dpi=300)

In [ ]:
# # Version from Daniel
# for label in all_labels:
#     vmin,vmax = np.nanpercentile(an_df_clustree.obs[label],1),np.nanpercentile(an_df_clustree.obs[label],99)
#     vcenter = median_vals_dict[label]
#     min_deviation,max_deviation = (vcenter-vmin,vmax-vcenter)
#     if min_deviation>max_deviation:
#         vmax = vcenter + min_deviation
#     else:
#         vmin = vcenter - max_deviation
#     min_vals.append(vmin)
#     max_vals.append(vmax)

# fig = sc.pl.umap(an_df_clustree, color=all_labels, show=False,\
#             title=title_list,legend_loc='on data', add_outline=False, size=50, vmin=min_vals, vmax=max_vals, \
#                  return_fig=True,cmap='RdBu_r',wspace=0.25,ncols=2, vcenter=[median_vals_dict[label] for label in all_labels])

# axes = fig.get_axes()
# for i,ax in enumerate(axes):
#     if i%2==1:
#         #colorbar settings
#         ax.set_ylabel(cbar_label,fontsize=16,labelpad=10)
#     else:
#         #plot settings
#         ax.set_title(ax.get_title(), fontsize=18)
#         ax.set_ylabel(ax.get_ylabel(), fontsize=18)
#         ax.set_xlabel(ax.get_xlabel(), fontsize=18)

# fig.savefig(umap_path + "/UMAP_Steady_State_Means.png",dpi=300)

### UMAP: Gene Overlay

In [ ]:
# Get ontologies
obo_fname = download_go_basic_obo()

# Get ecoli association file (ecocyc)
gaf_handle = goatools.base.http_get("http://current.geneontology.org/annotations/ecocyc.gaf.gz",fout="./ecocyc.gaf.gz")
gaf_fname = goatools.base.gunzip("./ecocyc.gaf.gz")

## Getting ontologies and other nonesense

obodag = GODag(obo_fname)
objanno = GafReader(gaf_fname)
ns2assoc = objanno.get_ns2assc()

gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
inv_gene_to_id = {assoc.DB_ID:assoc.DB_Symbol for assoc in objanno.associations}
synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
gene_to_id.update(synonym_dict)

In [ ]:
fig = highlight_sgrnas(an_df_clustree,an_df_clustree.obs[an_df_clustree.obs["Gene"].apply(lambda x: "fts" in x)].index.tolist())
axes = fig.get_axes()
for ax in axes:
    ax.set_title("fts Genes", fontsize=18)
    ax.set_ylabel(ax.get_ylabel(), fontsize=18)
    ax.set_xlabel(ax.get_xlabel(), fontsize=18)
# fig.savefig("./fts_genes.png",dpi=500)
fig.savefig(umap_path + "/UMAP_fts_genes.png",dpi=300)

In [ ]:
fig = highlight_sgrnas(an_df_clustree,an_df_clustree.obs[an_df_clustree.obs["Gene"].apply(lambda x: ("rps" in x)|("rpm" in x)|("rpl" in x))].index.tolist())
axes = fig.get_axes()
for ax in axes:
    ax.set_title("Ribosomal Protein Genes (rps,rpm,rpl)", fontsize=18)
    ax.set_ylabel(ax.get_ylabel(), fontsize=18)
    ax.set_xlabel(ax.get_xlabel(), fontsize=18)
# fig.savefig("./ribosomal_protein_genes.png",dpi=500)
fig.savefig(umap_path + "/UMAP_ribosomal_protein_genes.png",dpi=300)

In [ ]:
tRNA_aminoacylation_genes = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0043039")

fig = highlight_sgrnas(an_df_clustree,an_df_clustree.obs[an_df_clustree.obs["Gene"].apply(lambda x: x in tRNA_aminoacylation_genes)].index.tolist())
axes = fig.get_axes()
for ax in axes:
    ax.set_title("tRNA Aminoacetylation Genes (GO:0043039)", fontsize=18)
    ax.set_ylabel(ax.get_ylabel(), fontsize=18)
    ax.set_xlabel(ax.get_xlabel(), fontsize=18)
# fig.savefig("./tRNA_aminoacetylation_genes.png",dpi=500)
fig.savefig(umap_path + "/UMAP_tRNA_aminoacetylation_genes.png",dpi=300)

In [ ]:
test = an_df_clustree.obs
sec_ftsY_group = test[test["L5"]=='27']["Gene"]
ribostalk_group = test[test["L0"]=='6']["Gene"]

### UMAP: Cluster Timeseries

In [ ]:
def plot_cluster_timeseries(df, cluster_label, feature_labels, displayed_labels, feature_range_list, agg_fn=np.mean, x_ticks = [0,10,20], x_ticklabels=[0,5,10], cluster_subset=None, figsize=(10,10), wspace=0., hspace=0., fontsize=14, linewidth=5, color_list=None):

    if cluster_subset is not None:
        df = copy.copy(df)
        df = df[df[cluster_label].isin(cluster_subset)]
        
    timeseries_list = []
    for feature_label in feature_labels:        
        agg_cluster_timeseries = df.groupby([cluster_label]).apply(lambda x: agg_fn(np.array(x[feature_label].tolist()), axis=0)).to_frame()
        agg_cluster_timeseries = agg_cluster_timeseries.rename(columns={0:feature_label})
        timeseries_list.append(agg_cluster_timeseries)
    timeseries_df = pd.concat(timeseries_list, axis=1)
        
    fig = plt.figure(constrained_layout=True, figsize=figsize)
    gs = fig.add_gridspec(1,len(timeseries_df), wspace=wspace)

    for i in range(len(timeseries_df)):
        mean_cluster_timeseries = timeseries_df.iloc[i]
        
        clust_arr = np.array(timeseries_df.iloc[i].tolist())
        if color_list == None:
            color = "tab:blue"
        else:
            color = color_list[i%len(color_list)]
        
        if i == 0:
            inner_gs = gs[0,i].subgridspec(clust_arr.shape[0], 1, wspace=0, hspace=hspace)
            inner_grid_sub = inner_gs.subplots(sharex=True)
            for c, ax in np.ndenumerate(inner_grid_sub):
                
                feature_range = feature_range_list[c[0]]
                ax.plot(clust_arr[c],linewidth=linewidth,color=color)
                ax.set_ylim(feature_range[0],feature_range[1])
                ax.set(xticks=[])
#                 ax.set(xticks=[], yticks=[0,6])
                ax.set_ylabel(displayed_labels[c[0]],rotation=0,labelpad=30,fontsize=fontsize,ha="right")#,orientation="horizontal")
                
            ax.set_xlabel(str(i), fontsize=fontsize)
            ax.set(xticks=x_ticks)
            ax.set(xticklabels=x_ticklabels)
            
        else:
            inner_gs = gs[0,i].subgridspec(clust_arr.shape[0], 1, wspace=0, hspace=hspace)
            inner_grid_sub = inner_gs.subplots(sharex=True)
            for c, ax in np.ndenumerate(inner_grid_sub):
                feature_range = feature_range_list[c[0]]
                ax.plot(clust_arr[c],linewidth=linewidth,color=color)
                ax.set_ylim(feature_range[0],feature_range[1])
                ax.set(xticks=[], yticks=[])
                
            ax.set_xlabel(str(i), fontsize=fontsize)
            ax.set(xticks=x_ticks)
            ax.set(xticklabels=x_ticklabels)


    plt.tight_layout()
    return fig

In [ ]:
label_list = ['Mean: Delta time (hr)', # Changed from h
          'Mean: Septum Displacement Length Normalized',
          'Mean: major_axis_length',
          'Mean: minor_axis_length',
          'Mean: mCherry mean_intensity',
          'Mean: Number of Nucleoids',
          'Mean: YFP mean_intensity',
          'Mean: Total Nucleoid Area',
          'Mean: Nucleoid Area Mean',
          'Mean: Nucleoid Mean Intensity Mean',
          'Mean: Nucleoid Major Axis Length Mean',
          'Mean: Nucleoid Minor Axis Length Mean', 
          'Mean: Instantaneous Growth Rate: Volume']

In [ ]:
vmin, vmax

In [ ]:
range_list = [(vmin[i], vmax[i]) for i in range(len(vmin))]

In [ ]:
# LUIS
label_list = ['Mean: Delta time (hr)', # Changed from h
          'Mean: Septum Displacement Length Normalized',
          'Mean: major_axis_length',
          'Mean: minor_axis_length',
          'Mean: mCherry mean_intensity',
          'Mean: Number of Nucleoids',
          'Mean: YFP mean_intensity',
          'Mean: Total Nucleoid Area',
          'Mean: Nucleoid Area Mean',
          'Mean: Nucleoid Mean Intensity Mean',
          'Mean: Nucleoid Major Axis Length Mean',
          'Mean: Nucleoid Minor Axis Length Mean', 
          'Mean: Instantaneous Growth Rate: Volume']
title_list = label_list

# DANIEL
# label_list = ['Kernel Trace: Delta time (hr)','Kernel Trace: Septum Displacement Length Normalized','Kernel Trace: major_axis_length',\
# 'Kernel Trace: minor_axis_length','Kernel Trace: mCherry mean_intensity','Kernel Trace: Instantaneous Growth Rate: Volume']
# title_list = ["Interdivision Time (hr)","Septum Displacement/Length","Length $(\mu m)$","Width $(\mu m)$",\
#               "mKate2 Mean Intensity (AU)","Growth Rate (1/hr)"]
# range_list = []
# for i,label in enumerate(label_list):
#     vmin = np.percentile(np.array(an_df_clustree.obs[label].tolist()),0.1)
#     vmax = np.percentile(np.array(an_df_clustree.obs[label].tolist()),99.9)
#     range_list.append((vmin,vmax))


fig = plot_cluster_timeseries(an_df_clustree.obs,"L1", label_list,title_list,range_list,figsize=(40,12), x_ticks = [0,10,20], x_ticklabels=[0,5,10] ,wspace=0.25,hspace=0.25,color_list=vega_20_scanpy)
fig.savefig(umap_path + "/Leiden1_Cluster_Timeseries.png",dpi=300)

In [ ]:
label_list = ['Kernel Trace: Delta time (s): Transformed: z score','Kernel Trace: Septum Displacement Length Normalized: Transformed: z score','Kernel Trace: major_axis_length: Transformed: z score',\
'Kernel Trace: minor_axis_length: Transformed: z score','Kernel Trace: mCherry mean_intensity: Transformed: z score','Kernel Trace: Instantaneous Growth Rate: Volume: Transformed: z score']
title_list = ["Interdivision Time Z-score","Septum Displacement/Length Z-score","Length Z-score","Width Z-score",\
              "mKate2 Mean Intensity Z-score","Growth Rate Z-score"]
range_list = []
for i,label in enumerate(label_list):
    vmin = np.percentile(np.array(an_df_clustree.obs[label].tolist()),0.1)
    vmax = np.percentile(np.array(an_df_clustree.obs[label].tolist()),99.9)
    range_list.append((vmin,vmax))

fig = plot_cluster_timeseries(an_df_clustree.obs,"L1", label_list,title_list,range_list,figsize=(20,6), x_ticks = [0,10,20], x_ticklabels=[0,5,10], wspace=0.25,hspace=0.25,color_list=vega_20_scanpy)
fig.savefig(umap_path + "/Leiden1_Cluster_Timeseries_Zscores.png",dpi=300)

### Per-Cluster Metrics Not Added Earlier

- Silhouette Score
- Top two genes in each cluster (for clustree)

In [ ]:
for i,resolution in enumerate(resolution_list):
    silhouette_sample_labels = skl.metrics.silhouette_samples(norm_soft_dtw_arr, an_df_clustree.obs["L" + str(i)], metric='precomputed')
    an_df_clustree.obs["Silhouette_Score_" + "L" + str(i)] = silhouette_sample_labels

    top_2_genes_series = an_df_clustree.obs.groupby("L" + str(i)).apply(lambda x: "\n".join(get_top_genes(x["Gene"])))
    top_2_genes_df = pd.DataFrame(top_2_genes_series).rename(columns={0:"Top_2_Genes_" + "L" + str(i)})
    top_2_genes_df_columns = top_2_genes_df.columns
    an_df_clustree_addition = an_df_clustree.obs[["L" + str(i)]].reset_index().merge(top_2_genes_df, on="L" + str(i)).set_index('oDEPool7_id')
    an_df_clustree.obs[top_2_genes_df_columns] = an_df_clustree_addition[top_2_genes_df_columns]

## Clustree Analysis

### Clustree Visualization

In [ ]:
def convert_name_to_r(name,forbidden_characters=[":",")","(","-"]):
    name = name.replace(" ","_")
    for forbidden_character in forbidden_characters:
        name = name.replace(forbidden_character,"")
    return name

#### Convert to R-friendly Format

In [ ]:
import anndata2ri
import re
anndata2ri.activate()
%load_ext rpy2.ipython

real_valued_labels = ['Mean: Delta time (hr)','Mean: Septum Displacement Length Normalized','Mean: major_axis_length',\
          'Mean: minor_axis_length','Mean: mCherry mean_intensity','Mean: Instantaneous Growth Rate: Volume']
zscore_labels = ['Delta time (hr)','Septum Displacement Length Normalized','major_axis_length',\
          'minor_axis_length','mCherry mean_intensity','Instantaneous Growth Rate: Volume']

non_real_valued_labels = [name+": Integrated Z-score" for name in zscore_labels]+\
                        [name+": Last Timepoint Z-score" for name in zscore_labels]+\
                        [name+": Max Absolute Z-score" for name in zscore_labels]+\
                        [name+": Max Magnitude Z-score" for name in zscore_labels]+\
                        [name+": Max Magnitude Minus Last Timepoint Z-score" for name in zscore_labels]

real_valued_labels_renamed = [convert_name_to_r(name) for name in real_valued_labels]
non_real_valued_labels_renamed = [convert_name_to_r(name) for name in non_real_valued_labels]

labels_of_interest = real_valued_labels + non_real_valued_labels
labels_of_interest_renamed = real_valued_labels_renamed+non_real_valued_labels_renamed

cluster_column_names = [column_name for column_name in an_df_clustree.obs.columns if re.fullmatch("L[0-9]{0,100}",column_name)!=None]
cluster_property_list = [column_name for column_name in an_df_clustree.obs.columns if re.search("_L[0-9]{0,100}",column_name)!=None]
columns_to_fetch = ["Gene"] + labels_of_interest + cluster_column_names + cluster_property_list

an_df_clustree_r_compat = copy.deepcopy(an_df_clustree)
an_df_clustree_r_compat.obs = an_df_clustree.obs[columns_to_fetch]
an_df_clustree_r_compat.obs = an_df_clustree_r_compat.obs.rename(columns=dict(zip(labels_of_interest,labels_of_interest_renamed)))
an_df_clustree_non_r_compat = copy.deepcopy(an_df_clustree_r_compat)

median_vals_dict = control_ss_mean_cv_df[real_valued_labels].median().to_dict()

for key in list(an_df_clustree_r_compat.uns.keys()):
    del an_df_clustree_r_compat.uns[key]
for key in list(an_df_clustree_r_compat.obsp.keys()):
    del an_df_clustree_r_compat.obsp[key]

### Filtering out NaNs (hack)

In [ ]:
an_df_clustree_r_compat = an_df_clustree_r_compat[~np.any(np.array(an_df_clustree_r_compat.obs[labels_of_interest_renamed].isna().values.tolist()),axis=1)]

### Cleaning Up Dataframe

In [ ]:
columns_for_small_df = [
 'Kernel Trace: Delta time (s): Transformed: z score',
 'Kernel Trace: Septum Displacement Length Normalized: Transformed: z score',
 'Kernel Trace: Delta time (s)',
 'Kernel Trace: Septum Displacement',
 'Kernel Trace: Septum Displacement Length Normalized',
 'Kernel Trace: major_axis_length: Transformed: z score',
 'Kernel Trace: minor_axis_length: Transformed: z score',
 'Kernel Trace: mCherry mean_intensity: Transformed: z score',
 'Kernel Trace: major_axis_length',
 'Kernel Trace: minor_axis_length',
 'Kernel Trace: mCherry mean_intensity',
 'Kernel Trace: Instantaneous Growth Rate: Volume: Transformed: z score',
 'Kernel Trace: Instantaneous Growth Rate: Volume',
 'sgRNA',
 'Gene',
 'N Mismatch',
 'Category',
 'TargetID',
 'N Observations',
 'Feature Vector',
 'Kernel Trace: Delta time (hr)',
 'Integrated Z-scores',
 'Last Timepoint Z-scores',
 'Delta time (hr): Integrated Z-score',
 'Delta time (hr): Last Timepoint Z-score',
 'Septum Displacement Length Normalized: Integrated Z-score',
 'Septum Displacement Length Normalized: Last Timepoint Z-score',
 'major_axis_length: Integrated Z-score',
 'major_axis_length: Last Timepoint Z-score',
 'minor_axis_length: Integrated Z-score',
 'minor_axis_length: Last Timepoint Z-score',
 'mCherry mean_intensity: Integrated Z-score',
 'mCherry mean_intensity: Last Timepoint Z-score',
 'Instantaneous Growth Rate: Volume: Integrated Z-score',
 'Instantaneous Growth Rate: Volume: Last Timepoint Z-score',
 'Integrated Z-scores',
 'Feature Max Magnitude Z-score',
 'Mean: Septum Displacement',
 'Mean: Septum Displacement Length Normalized',
 'Mean: Delta time (s)',
 'Mean: major_axis_length',
 'Mean: minor_axis_length',
 'Mean: mCherry mean_intensity',
 'Mean: Instantaneous Growth Rate: Volume',
 'Mean: Delta time (hr)',
# 'Mean: GFP-Penta mean_intensity',
# 'Mean: GFP-Penta mean_intensity: Yeo-Johnson: z score',
 'L0',
 'L1',
 'L2',
 'L3',
 'L4',
 'L5',
 'Target Sequence',
 'N Target Sites',
 'Target Sites']

rename_column_dict = {'Kernel Trace: Delta time (s): Transformed: z score':'Interdivision Time (s) Z-score Timeseries',\
 'Kernel Trace: Septum Displacement Length Normalized: Transformed: z score':'Septum Displacement Length Normalized Z-score Timeseries',\
 'Kernel Trace: Delta time (s)':'Interdivision Time (s) Timeseries',\
 'Kernel Trace: Septum Displacement':'Septum Displacement (um) Timeseries',\
 'Kernel Trace: Septum Displacement Length Normalized':'Septum Displacement Length Normalized Timeseries',\
 'Kernel Trace: major_axis_length: Transformed: z score':'Length Z-score Timeseries',\
 'Kernel Trace: minor_axis_length: Transformed: z score':'Width Z-score Timeseries',\
 'Kernel Trace: mCherry mean_intensity: Transformed: z score':'mKate2 Mean Intensity Z-score Timeseries',\
 'Kernel Trace: major_axis_length':'Length (um) Timeseries',\
 'Kernel Trace: minor_axis_length':'Width (um) Timeseries',\
 'Kernel Trace: mCherry mean_intensity':'mKate2 Mean Intensity (AU) Timeseries',\
 'Kernel Trace: Instantaneous Growth Rate: Volume: Transformed: z score':'Volume Growth Rate Z-score Timeseries',\
 'Kernel Trace: Instantaneous Growth Rate: Volume':'Volume Growth Rate (1/hr) Timeseries',\
 'Kernel Trace: Delta time (hr)':'Interdivision Time (hr) Timeseries',\
 'Delta time (hr): Integrated Z-score':'Interdivision Time (hr) Integrated Z-score',\
 'Delta time (hr): Last Timepoint Z-score':'Interdivision Time (hr) Last Timepoint Z-score',\
 'Septum Displacement Length Normalized: Integrated Z-score':'Septum Displacement Length Normalized Integrated Z-score',\
 'Septum Displacement Length Normalized: Last Timepoint Z-score':'Septum Displacement Length Normalized Last Timepoint Z-score',\
 'major_axis_length: Integrated Z-score':'Length Integrated Z-score',\
 'major_axis_length: Last Timepoint Z-score':'Length Last Timepoint Z-score',\
 'minor_axis_length: Integrated Z-score':'Width Integrated Z-score',\
 'minor_axis_length: Last Timepoint Z-score':'Width Last Timepoint Z-score',\
 'mCherry mean_intensity: Integrated Z-score':'mKate2 Mean Intensity Integrated Z-score',\
 'mCherry mean_intensity: Last Timepoint Z-score':'mKate2 Mean Intensity Last Timepoint Z-score',\
 'Instantaneous Growth Rate: Volume: Integrated Z-score':'Volume Growth Rate Integrated Z-score',\
 'Instantaneous Growth Rate: Volume: Last Timepoint Z-score':'Volume Growth Rate Last Timepoint Z-score',\
 'Mean: Septum Displacement':'Mean Septum Displacement (um) Steady State',\
 'Mean: Septum Displacement Length Normalized':'Mean Septum Displacement Length Normalized Steady State',\
 'Mean: Delta time (s)':'Mean Interdivision Time (s) Steady State',\
 'Mean: major_axis_length':'Mean Length (um) Steady State',\
 'Mean: minor_axis_length':'Mean Width (um) Steady State',\
 'Mean: mCherry mean_intensity':'Mean mKate2 Mean Intensity (AU) Steady State',\
 'Mean: Instantaneous Growth Rate: Volume':'Mean Volume Growth Rate (1/hr) Steady State',\
 'Mean: Delta time (hr)':'Mean Interdivision Time (hr) Steady State',\
 'Mean: GFP-Penta mean_intensity':'Mean GFP Mean Intensity (AU) (at $t_{f}$ for nucleoid snapshots)',\
 'Mean: GFP-Penta mean_intensity: Yeo-Johnson: z score':'Mean GFP Mean Intensity Z-score (at $t_{f}$ for nucleoid snapshots)'}

big_output_df = an_df_clustree.obs
big_output_df.index = an_df_clustree.obs.index.astype(int)
big_output_df = big_output_df.merge(lDE23_design[["Target Sequence","N Target Sites","Target Sites"]],how="left",left_on="oDEPool7_id",right_index=True)

small_output_df = big_output_df[columns_for_small_df]
small_output_df = small_output_df.rename(columns=rename_column_dict)

very_small_output_df = small_output_df.drop(columns=[name for name in small_output_df.columns if "Timeseries" in name]+["Feature Vector","Integrated Z-scores","Last Timepoint Z-scores","Feature Max Magnitude Z-score"])

### Save Clustree Dataframe - Only Pandas Dataframe (pickle and csv)

In [ ]:
big_output_df.to_pickle(z_score_thr_dir+"/Pandas_Dataframe.pkl")
small_output_df.to_pickle(z_score_thr_dir+"/Pandas_Dataframe_Small.pkl")
very_small_output_df.to_pickle(z_score_thr_dir+"/Pandas_Dataframe_Very_Small.pkl")

big_output_df.to_csv(z_score_thr_dir+"/Pandas_Dataframe.csv")
small_output_df.to_csv(z_score_thr_dir+"/Pandas_Dataframe_Small.csv")
very_small_output_df.to_csv(z_score_thr_dir+"/Pandas_Dataframe_Very_Small.csv")

### Save Clustree R-compatible AnnData (hdf5)

In [ ]:
an_df_clustree_r_compat.write(z_score_thr_dir+"/AnnData.h5ad")
an_df_clustree_non_r_compat.write(z_score_thr_dir+"/AnnData_nonRcompat.h5ad")

# LUIS: FINISH HERE

### Reload R-compatible AnnData and Clustree Pandas Dataframe

In [ ]:
clustree_pandas_df = pd.read_pickle(z_score_thr_dir+"/Pandas_Dataframe.pkl")
an_df_clustree_r_compat = anndata.read_h5ad(z_score_thr_dir+"/AnnData.h5ad")

### Plot Cluster Metrics

In [ ]:
recall_metrics = ["Gene","TargetID"]
label_to_plot = "Top_2_Genes_"
recall_values_to_plot = ["Max_Recall_","Top2_Recall_","Median_Recall_"]
values_to_plot = [metric+"_"+name for name in recall_values_to_plot for metric in recall_metrics]

In [ ]:
%%R -i an_df_clustree_r_compat,values_to_plot,clustree_path,label_to_plot
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
scale_fill_brewer_c <- function(...) {
        scale_fill_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)
options(ggplot2.continuous.fill=scale_fill_brewer_c)

assayNames(an_df_clustree_r_compat) <- "counts"
prefix <- "L"

label_prefix <- paste(label_to_plot,prefix,sep="")

for (value_to_plot in values_to_plot) {

    counter <- 0
    parameter_prefix <- paste(value_to_plot,prefix,sep="")
    
    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }
    
    fetchlabel <- function(values) {
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_r_compat@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    cols_label_prefix <- substr(colnames(x), 1, nchar(label_prefix))
    
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix
    label_cols <- cols_label_prefix == label_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    label_matrix <- as.matrix(x[, label_cols])
    param_listoflists <- apply(param_matrix, 1, as.list)
    label_listoflists <- apply(label_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_r_compat$paramconcat = param_listoflists
    an_df_clustree_r_compat$labelconcat = label_listoflists

    ggpath <- paste(clustree_path,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_r_compat, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", 
                         node_colour_aggr="fetchparam",node_label="labelconcat",node_label_aggr="fetchlabel",node_label_size = 2) + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, dpi=100, width=42, height=10)
    
}

In [ ]:
label_to_plot = "Top_2_Genes_"
values_to_plot = ["Silhouette_Score_"]

In [ ]:
%%R -i an_df_clustree_r_compat,values_to_plot,clustree_path,label_to_plot
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="div",palette="RdBu",direction=-1,limit=c(-1.,1.),oob=squish,...)
    }
scale_fill_brewer_c <- function(...) {
        scale_fill_distiller(type="div",palette="RdBu",direction=-1,limit=c(-1.,1.),oob=squish,...)
    }

options(ggplot2.continuous.colour=scale_colour_brewer_c)
options(ggplot2.continuous.fill=scale_fill_brewer_c)

assayNames(an_df_clustree_r_compat) <- "counts"
label_prefix <- paste(label_to_plot,prefix,sep="")

for (value_to_plot in values_to_plot) {

    counter <- 0
    prefix <- "L"
    parameter_prefix <- paste(value_to_plot,prefix,sep="")

    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }
    
    fetchlabel <- function(values) {
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_r_compat@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    cols_label_prefix <- substr(colnames(x), 1, nchar(label_prefix))
    
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix
    label_cols <- cols_label_prefix == label_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    label_matrix <- as.matrix(x[, label_cols])
    
    param_listoflists <- apply(param_matrix, 1, as.list)
    label_listoflists <- apply(label_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_r_compat$paramconcat = param_listoflists
    an_df_clustree_r_compat$labelconcat = label_listoflists

    ggpath <- paste(clustree_path,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_r_compat, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", node_colour_aggr="fetchparam",
                        node_label="labelconcat",node_label_aggr="fetchlabel",node_label_size = 2) + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, dpi=100, width=42, height=10)
    
}

### Plot Cluster Properties (with real values)

In [ ]:
values_to_plot = real_valued_labels_renamed
median_vals_dict = control_ss_mean_cv_df[real_valued_labels].median().to_dict()
median_vals_key_list_for_r = [item[0].replace(" ","_").replace(":","").replace(")","").replace("(","") for item in median_vals_dict.items()]
median_vals_val_list_for_r = [item[1] for item in median_vals_dict.items()]

In [ ]:
%%R -i an_df_clustree_r_compat,values_to_plot,clustree_path,median_vals_key_list_for_r,median_vals_val_list_for_r
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library("ggplot2")

names(median_vals_val_list_for_r) <- median_vals_key_list_for_r

define_cmap_for_real_values <- function(clustree_df,value_to_plot,median_vals_val_list_for_r) {
    vmin <- quantile(clustree_df[[value_to_plot]], c(0.01))[[1]]
    vmax <- quantile(clustree_df[[value_to_plot]], c(0.99))[[1]]
    vcenter <- median_vals_val_list_for_r[[value_to_plot]]
    min_deviation <- vcenter-vmin
    max_deviation <- vmax-vcenter
    if (min_deviation>max_deviation) {
        vmax <- vcenter + min_deviation
    } else {
        vmin <- vcenter - max_deviation
    }
    scale_colour_brewer_c <<- function(...) {
        scale_colour_distiller(type="div",palette="RdBu",direction=-1,limit=c(vmin,vmax),oob=squish,...)
    }
}

# options(ggplot2.continuous.colour=scale_colour_brewer(type="div",palette="RdBu",direction=1,aesthetics = "colour"))
# options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_r_compat) <- "counts"
for (value_to_plot in values_to_plot) {
    ggpath <- paste(clustree_path,"/",value_to_plot,"_","ggplot.png",sep="")
    
    define_cmap_for_real_values(an_df_clustree_r_compat,value_to_plot,median_vals_val_list_for_r)
    options(ggplot2.continuous.colour=scale_colour_brewer_c)
    
    clustplot <- clustree(an_df_clustree_r_compat, prefix = prefix, node_size_range = c(7, 20), node_colour=value_to_plot, node_colour_aggr="mean") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    }
clustplot

### Plot Cluster Properties (with Z scores)

In [ ]:
values_to_plot = non_real_valued_labels_renamed

In [ ]:
%%R -i an_df_clustree_r_compat,values_to_plot,clustree_path
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="div",palette="RdBu",direction=-1,limit=c(-2.,2.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_r_compat) <- "counts"
for (value_to_plot in values_to_plot) {
    ggpath <- paste(clustree_path,"/",value_to_plot,"_","ggplot.png",sep="")    
    clustplot <- clustree(an_df_clustree_r_compat, prefix = prefix, node_size_range = c(7, 20), node_colour=value_to_plot, node_colour_aggr="mean") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    }
clustplot

### Examine Gene Distrubtion

#### Ribosomal Proteins (rps,rpl,rpm)

In [ ]:
an_df_clustree_gene = copy.deepcopy(an_df_clustree_r_compat)

gene_regex = "rp[sml]"
gene_name = "rps_rpm_rpl"
gene_dir = clustree_path + "/Gene_" + gene_name
tr.writedir(gene_dir, overwrite=False)

for i,resolution in enumerate(resolution_list):
    size_series = an_df_clustree_gene.obs.groupby("L" + str(i)).size()
    gene_count_series = an_df_clustree_gene.obs.groupby("L" + str(i)).apply(lambda x:np.sum([re.search(gene_regex,gene) != None for gene in x["Gene"].tolist()]))
    gene_frequency_series = gene_count_series/size_series
    gene_perc_of_total_series = gene_count_series/np.sum(gene_count_series)
    
    gene_df = pd.concat([gene_frequency_series,gene_perc_of_total_series],axis=1).rename(columns={0:"Percent_Of_Cluster_" + "L" + str(i),1:"Percent_Of_Gene_" + "L" + str(i)})
    gene_df_columns = gene_df.columns
    
    an_df_clustree_gene_addition = an_df_clustree_gene.obs[["L" + str(i)]].reset_index().merge(gene_df, on="L" + str(i)).set_index('oDEPool7_id')
    an_df_clustree_gene.obs[gene_df_columns] = an_df_clustree_gene_addition[gene_df_columns]

In [ ]:
values_to_plot = ["Percent_Of_Cluster_","Percent_Of_Gene_"]

In [ ]:
%%R -i an_df_clustree_gene,values_to_plot,gene_dir
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_gene) <- "counts"

for (value_to_plot in values_to_plot) {

    counter <- 0
    prefix <- "L"
    parameter_prefix <- paste(value_to_plot,prefix,sep="")

    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_gene@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    param_listoflists <- apply(param_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_gene$paramconcat = param_listoflists

    ggpath <- paste(gene_dir,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_gene, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", node_colour_aggr="fetchparam") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    
}

#### Divisome Proteins (fts)

In [ ]:
an_df_clustree_gene = copy.deepcopy(an_df_clustree_r_compat)

gene_regex = "fts"
gene_name = "fts"
gene_dir = clustree_path + "/Gene_" + gene_name
tr.writedir(gene_dir, overwrite=False)

for i,resolution in enumerate(resolution_list):
    size_series = an_df_clustree_gene.obs.groupby("L" + str(i)).size()
    gene_count_series = an_df_clustree_gene.obs.groupby("L" + str(i)).apply(lambda x:np.sum([re.search(gene_regex,gene) != None for gene in x["Gene"].tolist()]))
    gene_frequency_series = gene_count_series/size_series
    gene_perc_of_total_series = gene_count_series/np.sum(gene_count_series)
    
    gene_df = pd.concat([gene_frequency_series,gene_perc_of_total_series],axis=1).rename(columns={0:"Percent_Of_Cluster_" + "L" + str(i),1:"Percent_Of_Gene_" + "L" + str(i)})
    gene_df_columns = gene_df.columns
    
    an_df_clustree_gene_addition = an_df_clustree_gene.obs[["L" + str(i)]].reset_index().merge(gene_df, on="L" + str(i)).set_index('oDEPool7_id')
    an_df_clustree_gene.obs[gene_df_columns] = an_df_clustree_gene_addition[gene_df_columns]

In [ ]:
values_to_plot = ["Percent_Of_Cluster_","Percent_Of_Gene_"]

In [ ]:
%%R -i an_df_clustree_gene,values_to_plot,gene_dir
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_gene) <- "counts"

for (value_to_plot in values_to_plot) {

    counter <- 0
    prefix <- "L"
    parameter_prefix <- paste(value_to_plot,prefix,sep="")

    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_gene@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    param_listoflists <- apply(param_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_gene$paramconcat = param_listoflists

    ggpath <- paste(gene_dir,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_gene, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", node_colour_aggr="fetchparam") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    
}

#### tRNA Aminoacetylation Genes

In [ ]:
# Get ontologies
obo_fname = download_go_basic_obo()

# Get ecoli association file (ecocyc)
gaf_handle = goatools.base.http_get("http://current.geneontology.org/annotations/ecocyc.gaf.gz",fout="./ecocyc.gaf.gz")
gaf_fname = goatools.base.gunzip("./ecocyc.gaf.gz")

## Getting ontologies and other nonesense

obodag = GODag(obo_fname)
objanno = GafReader(gaf_fname)
ns2assoc = objanno.get_ns2assc()

gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
inv_gene_to_id = {assoc.DB_ID:assoc.DB_Symbol for assoc in objanno.associations}
synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
gene_to_id.update(synonym_dict)

In [ ]:
def search_go(ns2assoc, obodag, inv_gene_to_id, go_term):
    namespace_abbv = {"biological_process":"BP","molecular_function":"MF","cellular_component":"CC"}
    
    print("Searching for " + str(obodag[go_term].name))
    namespace = namespace_abbv[obodag[go_term].namespace]
    child_goterms = list(obodag[go_term].get_all_children())
    gene_list = [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if go_term in val]
    for child_goterm in child_goterms:    
        gene_list += [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if child_goterm in val]
    gene_list = sorted(list(set(gene_list)))
    return gene_list

In [ ]:
an_df_clustree_gene = copy.deepcopy(an_df_clustree_r_compat)

tRNA_aminoacylation_genes = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0043039")
gene_regex = r"(?=("+'|'.join(tRNA_aminoacylation_genes)+r"))"
gene_name = "tRNA_aminoacetylation"
gene_dir = clustree_path + "/Gene_" + gene_name
tr.writedir(gene_dir, overwrite=False)

for i,resolution in enumerate(resolution_list):
    size_series = an_df_clustree_gene.obs.groupby("L" + str(i)).size()
    gene_count_series = an_df_clustree_gene.obs.groupby("L" + str(i)).apply(lambda x:np.sum([re.search(gene_regex,gene) != None for gene in x["Gene"].tolist()]))
    gene_frequency_series = gene_count_series/size_series
    gene_perc_of_total_series = gene_count_series/np.sum(gene_count_series)
    
    gene_df = pd.concat([gene_frequency_series,gene_perc_of_total_series],axis=1).rename(columns={0:"Percent_Of_Cluster_" + "L" + str(i),1:"Percent_Of_Gene_" + "L" + str(i)})
    gene_df_columns = gene_df.columns
    
    an_df_clustree_gene_addition = an_df_clustree_gene.obs[["L" + str(i)]].reset_index().merge(gene_df, on="L" + str(i)).set_index('oDEPool7_id')
    an_df_clustree_gene.obs[gene_df_columns] = an_df_clustree_gene_addition[gene_df_columns]

In [ ]:
values_to_plot = ["Percent_Of_Cluster_","Percent_Of_Gene_"]

In [ ]:
%%R -i an_df_clustree_gene,values_to_plot,gene_dir
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_gene) <- "counts"

for (value_to_plot in values_to_plot) {

    counter <- 0
    prefix <- "L"
    parameter_prefix <- paste(value_to_plot,prefix,sep="")

    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_gene@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    param_listoflists <- apply(param_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_gene$paramconcat = param_listoflists

    ggpath <- paste(gene_dir,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_gene, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", node_colour_aggr="fetchparam") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    
}

#### Min System

In [ ]:
an_df_clustree_gene = copy.deepcopy(an_df_clustree_r_compat)

gene_regex = "min"
gene_name = "min"
gene_dir = clustree_path + "/Gene_" + gene_name
tr.writedir(gene_dir, overwrite=False)

for i,resolution in enumerate(resolution_list):
    size_series = an_df_clustree_gene.obs.groupby("L" + str(i)).size()
    gene_count_series = an_df_clustree_gene.obs.groupby("L" + str(i)).apply(lambda x:np.sum([re.search(gene_regex,gene) != None for gene in x["Gene"].tolist()]))
    gene_frequency_series = gene_count_series/size_series
    gene_perc_of_total_series = gene_count_series/np.sum(gene_count_series)
    
    gene_df = pd.concat([gene_frequency_series,gene_perc_of_total_series],axis=1).rename(columns={0:"Percent_Of_Cluster_" + "L" + str(i),1:"Percent_Of_Gene_" + "L" + str(i)})
    gene_df_columns = gene_df.columns
    
    an_df_clustree_gene_addition = an_df_clustree_gene.obs[["L" + str(i)]].reset_index().merge(gene_df, on="L" + str(i)).set_index('oDEPool7_id')
    an_df_clustree_gene.obs[gene_df_columns] = an_df_clustree_gene_addition[gene_df_columns]

In [ ]:
values_to_plot = ["Percent_Of_Cluster_","Percent_Of_Gene_"]

In [ ]:
%%R -i an_df_clustree_gene,values_to_plot,gene_dir
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_gene) <- "counts"

for (value_to_plot in values_to_plot) {

    counter <- 0
    prefix <- "L"
    parameter_prefix <- paste(value_to_plot,prefix,sep="")

    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_gene@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    param_listoflists <- apply(param_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_gene$paramconcat = param_listoflists

    ggpath <- paste(gene_dir,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_gene, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", node_colour_aggr="fetchparam") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    
}

#### fab Genes

In [ ]:
an_df_clustree_gene = copy.deepcopy(an_df_clustree_r_compat)

gene_regex = "fab"
gene_name = "fab"
gene_dir = clustree_path + "/Gene_" + gene_name
tr.writedir(gene_dir, overwrite=False)

for i,resolution in enumerate(resolution_list):
    size_series = an_df_clustree_gene.obs.groupby("L" + str(i)).size()
    gene_count_series = an_df_clustree_gene.obs.groupby("L" + str(i)).apply(lambda x:np.sum([re.search(gene_regex,gene) != None for gene in x["Gene"].tolist()]))
    gene_frequency_series = gene_count_series/size_series
    gene_perc_of_total_series = gene_count_series/np.sum(gene_count_series)
    
    gene_df = pd.concat([gene_frequency_series,gene_perc_of_total_series],axis=1).rename(columns={0:"Percent_Of_Cluster_" + "L" + str(i),1:"Percent_Of_Gene_" + "L" + str(i)})
    gene_df_columns = gene_df.columns
    
    an_df_clustree_gene_addition = an_df_clustree_gene.obs[["L" + str(i)]].reset_index().merge(gene_df, on="L" + str(i)).set_index('oDEPool7_id')
    an_df_clustree_gene.obs[gene_df_columns] = an_df_clustree_gene_addition[gene_df_columns]

In [ ]:
values_to_plot = ["Percent_Of_Cluster_","Percent_Of_Gene_"]

In [ ]:
%%R -i an_df_clustree_gene,values_to_plot,gene_dir
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_gene) <- "counts"

for (value_to_plot in values_to_plot) {

    counter <- 0
    prefix <- "L"
    parameter_prefix <- paste(value_to_plot,prefix,sep="")

    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_gene@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    param_listoflists <- apply(param_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_gene$paramconcat = param_listoflists

    ggpath <- paste(gene_dir,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_gene, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", node_colour_aggr="fetchparam") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    
}

#### bam Genes

In [ ]:
an_df_clustree_gene = copy.deepcopy(an_df_clustree_r_compat)

gene_regex = "bam"
gene_name = "bam"
gene_dir = clustree_path + "/Gene_" + gene_name
tr.writedir(gene_dir, overwrite=False)

for i,resolution in enumerate(resolution_list):
    size_series = an_df_clustree_gene.obs.groupby("L" + str(i)).size()
    gene_count_series = an_df_clustree_gene.obs.groupby("L" + str(i)).apply(lambda x:np.sum([re.search(gene_regex,gene) != None for gene in x["Gene"].tolist()]))
    gene_frequency_series = gene_count_series/size_series
    gene_perc_of_total_series = gene_count_series/np.sum(gene_count_series)
    
    gene_df = pd.concat([gene_frequency_series,gene_perc_of_total_series],axis=1).rename(columns={0:"Percent_Of_Cluster_" + "L" + str(i),1:"Percent_Of_Gene_" + "L" + str(i)})
    gene_df_columns = gene_df.columns
    
    an_df_clustree_gene_addition = an_df_clustree_gene.obs[["L" + str(i)]].reset_index().merge(gene_df, on="L" + str(i)).set_index('oDEPool7_id')
    an_df_clustree_gene.obs[gene_df_columns] = an_df_clustree_gene_addition[gene_df_columns]

In [ ]:
values_to_plot = ["Percent_Of_Cluster_","Percent_Of_Gene_"]

In [ ]:
%%R -i an_df_clustree_gene,values_to_plot,gene_dir
## This is the main example of how to use clustree, with a fix to do node-wise aggregation
## Develop the earlier metrics and analysis a bit and plug in here
library(clustree)
library(scales)
library(RColorBrewer)
library("ggplot2")

scale_colour_brewer_c <- function(...) {
        scale_colour_distiller(type="seq",palette="Blues",values=c(0,0.166,0.333,0.5,0.666,0.833,1.,1.5,1.5),direction=1,limit=c(0.,1.),oob=squish,...)
    }
options(ggplot2.continuous.colour=scale_colour_brewer_c)

assayNames(an_df_clustree_gene) <- "counts"

for (value_to_plot in values_to_plot) {

    counter <- 0
    prefix <- "L"
    parameter_prefix <- paste(value_to_plot,prefix,sep="")

    fetchparam <- function(values) {
        counter <<- counter+1
        current_clustering_level <- which(cluster_length_cumulative_list>=counter)[[1]]
        # print(current_clustering_level)
        values_matrix <- do.call("cbind",values)
        output <- values_matrix[current_clustering_level,1]
        return(output)
    }

    #Converting the SnigleCellExperiment to the format clustree likes to use internally

    x <- data.frame(an_df_clustree_gene@colData)

    #Fetching a the list of clusters as they will get assigned withing clustplot
    cols_prefix <- substr(colnames(x), 1, nchar(prefix))
    cols_param_prefix <- substr(colnames(x), 1, nchar(parameter_prefix))
    clust_cols <- cols_prefix == prefix
    param_cols <- cols_param_prefix == parameter_prefix

    clusterings <- as.matrix(x[, clust_cols])
    param_matrix <- as.matrix(x[, param_cols])
    param_listoflists <- apply(param_matrix, 1, as.list)

    sorted_cluster_list <- lapply(colnames(clusterings), function(res) {
            clustering <- clusterings[, res]
            clusters <- sort(unique(clustering))
            return(clusters)
        })

    cluster_length_list <- lapply(sorted_cluster_list,length)
    cluster_length_cumulative_list <- cumsum(cluster_length_list)

    an_df_clustree_gene$paramconcat = param_listoflists

    ggpath <- paste(gene_dir,"/",value_to_plot,"_","ggplot.png",sep="")
    clustplot = clustree(an_df_clustree_gene, prefix = prefix, return="plot", node_size_range = c(7, 20), node_colour="paramconcat", node_colour_aggr="fetchparam") + 
                theme(legend.position = "bottom")
    ggsave(ggpath, plot = clustplot, scale = 2, dpi=100, width=24, height=7)
    
}

### Examine Timeseries

In [ ]:
def plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,n_hours_expt = 32,figsize = (24,12),nrows = 2,ncols = 3,\
                           labels_to_plot = ['Delta time (hr)','Septum Displacement Length Normalized','major_axis_length',\
                          'minor_axis_length','mCherry mean_intensity','Instantaneous Growth Rate: Volume']):

    labels_to_plot_map = {i:label for i,label in enumerate(labels_to_plot)}
    kernel_trace_labels_to_plot = ["Kernel Trace: " + label for label in labels_to_plot]
    kernel_trace_zscore_labels = ["Kernel Trace: " + label + ": Transformed: z score" for label in labels_to_plot]
    #hack to replace hr with s for zscore
    kernel_trace_zscore_labels = ['Kernel Trace: Delta time (s): Transformed: z score'] + kernel_trace_zscore_labels[1:]

    selected_df = clustree_pandas_df[clustree_pandas_df[cluster_level] == cluster_number]
    timseries_arr = np.array(selected_df[kernel_trace_labels_to_plot].values.tolist())
    timeseries_zscore_arr = np.array(selected_df[kernel_trace_zscore_labels].values.tolist())
    obs_grid_map = {i:time for i,time in enumerate(np.linspace(0,n_hours_expt,num=timseries_arr.shape[2]))}
    
    timseries_xarr = xr.DataArray(timseries_arr,dims=["sgRNA","Property","Time (hr)"])
    timseries_df = timseries_xarr.to_dataframe("Observable")
    timseries_df = timseries_df.reset_index()
    timseries_df["Property"] = timseries_df["Property"].apply(lambda x: labels_to_plot_map[x])
    timseries_df["Time (hr)"] = timseries_df["Time (hr)"].apply(lambda x: obs_grid_map[x])

    #temp using euclidean (will use z-scores)
    euc_dist_mat = skl.metrics.euclidean_distances(timeseries_zscore_arr.reshape(timeseries_zscore_arr.shape[0],-1))    
    max_different_timeseries = [np.where(euc_dist_mat==np.max(euc_dist_mat))[0][0],np.where(euc_dist_mat==np.max(euc_dist_mat))[1][0]]
    

    fig = plt.figure(figsize=figsize)
    for prop_idx in range(timseries_arr.shape[1]):
        plt.subplot(nrows,ncols,prop_idx+1)
        vmin = 0
        # vmin = np.percentile(np.array(clustree_pandas_df[kernel_trace_labels_to_plot[prop_idx]].tolist()),0.1)
        vmax = np.percentile(np.array(clustree_pandas_df[kernel_trace_labels_to_plot[prop_idx]].tolist()),99.9)

        timseries_df_single_prop = timseries_df[timseries_df["Property"] == labels_to_plot[prop_idx]]
        timseries_df_single_prop = timseries_df_single_prop.rename(columns={"Observable":labels_to_plot[prop_idx]})
        timseries_df_single_prop_max_different = timseries_df_single_prop[timseries_df_single_prop["sgRNA"].isin(max_different_timeseries)]
        if (prop_idx+1 == ncols):
            sns.lineplot(x="Time (hr)",y=labels_to_plot[prop_idx], data=timseries_df_single_prop, ci="sd",label="Mean Behavior")
            sns.lineplot(x="Time (hr)",y=labels_to_plot[prop_idx], data=timseries_df_single_prop_max_different, estimator=None, units="sgRNA",label="Most Different Timeseries")
        else:
            sns.lineplot(x="Time (hr)",y=labels_to_plot[prop_idx], data=timseries_df_single_prop, ci="sd")
            sns.lineplot(x="Time (hr)",y=labels_to_plot[prop_idx], data=timseries_df_single_prop_max_different, estimator=None, units="sgRNA")
        plt.ylim(vmin,vmax)
    return fig

In [ ]:
z_score_type = ["Integrated","90th Percentile","Max"][2]
strong_effect_threshold = [1.5,2.][1]
n_neighbors = [5,10,15][1]

rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-01-02_lDE23_32Hour_Analysis"
z_score_thr_dir = rootpath + "/Z_Score_Thr_" + str(strong_effect_threshold) + "_N_Neighbors_" + str(n_neighbors)
clustree_path = z_score_thr_dir + "/Clustree"
umap_path = z_score_thr_dir + "/UMAP"
compare_path = z_score_thr_dir + "/Comparison"

clustree_pandas_df = pd.read_pickle(z_score_thr_dir+"/Pandas_Dataframe.pkl")
an_df_clustree_r_compat = anndata.read_h5ad(z_score_thr_dir+"/AnnData.h5ad")

In [ ]:
cluster_level = "L0"
cluster_number = "7"

cluster_timeseries_path = clustree_path + "/Cluster_Timeseries"
tr.writedir(cluster_timeseries_path, overwrite=False)

fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,figsize=(18,10))
fig.savefig(cluster_timeseries_path + "/" + cluster_level + "_" + cluster_number + ".jpg", dpi=100)

In [ ]:
cluster_level = "L1"
cluster_number = "14"

cluster_timeseries_path = clustree_path + "/Cluster_Timeseries"
tr.writedir(cluster_timeseries_path, overwrite=False)

fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,figsize=(18,10))
fig.savefig(cluster_timeseries_path + "/" + cluster_level + "_" + cluster_number + ".jpg", dpi=100)

In [ ]:
cluster_level = "L0"
cluster_number = "6"

cluster_timeseries_path = clustree_path + "/Cluster_Timeseries"
tr.writedir(cluster_timeseries_path, overwrite=False)

fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,figsize=(18,10))
fig.savefig(cluster_timeseries_path + "/" + cluster_level + "_" + cluster_number + ".jpg", dpi=100)

In [ ]:
cluster_level = "L0"
cluster_number = "1"

cluster_timeseries_path = clustree_path + "/Cluster_Timeseries"
tr.writedir(cluster_timeseries_path, overwrite=False)

fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,figsize=(18,10))
fig.savefig(cluster_timeseries_path + "/" + cluster_level + "_" + cluster_number + ".jpg", dpi=100)

In [ ]:
cluster_level = "L0"
cluster_number = "4"

cluster_timeseries_path = clustree_path + "/Cluster_Timeseries"
tr.writedir(cluster_timeseries_path, overwrite=False)

fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,figsize=(18,10))
fig.savefig(cluster_timeseries_path + "/" + cluster_level + "_" + cluster_number + ".jpg", dpi=100)

In [ ]:
cluster_level = "L2"
cluster_number = "25"

cluster_timeseries_path = clustree_path + "/Cluster_Timeseries"
tr.writedir(cluster_timeseries_path, overwrite=False)

fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,figsize=(18,10))
fig.savefig(cluster_timeseries_path + "/" + cluster_level + "_" + cluster_number + ".jpg", dpi=100)

In [ ]:
cluster_level = "L3"
cluster_number = "20"

cluster_timeseries_path = clustree_path + "/Cluster_Timeseries"
tr.writedir(cluster_timeseries_path, overwrite=False)

fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_number,figsize=(18,10))
fig.savefig(cluster_timeseries_path + "/" + cluster_level + "_" + cluster_number + ".jpg", dpi=100)

### Test Violin Plot (Figure 2)

- Currently using kernel traces since some params dont have steady state values but do have kernel trace values

In [ ]:
cluster_levels = ["L0","L0","L0","L0","L2"]
clusters = ["5","3","1","4","25"]

labels = ['Delta time (hr)','Septum Displacement Length Normalized','major_axis_length',\
          'minor_axis_length','mCherry mean_intensity','Instantaneous Growth Rate: Volume']
kernel_labels = ['Kernel Trace: ' + label for label in labels]
cluster_names = ["Division","Wide","Starved","Ribosome","Ribosome Stalk"]
label_names = ["Interdivision Time (hr)","Septum Displacement","Long Axis Length ($\mu m$)",\
          "Short Axis Length ($\mu m$)","mKate2 Intensity (AU)","Growth Rate (1/hr)"]

merged_df = []
for i,cluster_level in enumerate(cluster_levels):
    selected_df = clustree_pandas_df[clustree_pandas_df[cluster_level]==clusters[i]]
    last_value_df = selected_df[kernel_labels].apply(lambda x: [kernel[-1] for kernel in x.tolist()], axis=0)
    last_value_df["Cluster Name"] = cluster_names[i]
    merged_df.append(last_value_df)
merged_df = pd.concat(merged_df)

In [ ]:
fig = plt.figure(figsize=(8,6))
for i in range(len(kernel_labels)):
    plt.subplot(2,3,i+1)
    plt.xticks(rotation=45)
    sns.violinplot(data=merged_df,x="Cluster Name",y=kernel_labels[i])
    plt.ylabel(label_names[i])
    plt.xlabel("")

plt.tight_layout()
plt.savefig(rootpath+"/violin_plot_mockup.jpg",dpi=300)

### Zip Current Clustree Folder

In [ ]:
import shutil

rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-01-02_lDE23_32Hour_Analysis"
dir_above = "/".join(rootpath.split("/")[:-1]) + "/"
rootpath_dir = rootpath.split("/")[-1]
shutil.make_archive(rootpath,"zip",root_dir=dir_above,base_dir=rootpath_dir)

### Notes:
- The fab and acc gene clusters (L3-13) and (L5-94) are very interesting in terms of compensation
- Generally, some compensation in the growth rate seems common, explore this further later
- Otherwise, no strong evidence of compensation using the Max(Z) - Z(tf) metric
- Consider finding other ways to find this:
    - Take a day to figure out other "compensation metrics"
- Determine next steps

In [ ]:
dask_controller.shutdown()